# Revisão JVOICE-D-26-01691 · Gravidade, idade e sexo (v4)

Responde aos pontos centrais do Revisor 1. Basta rodar de cima a baixo: o notebook encontra a matriz de features em `MyDrive/svd_data`, recupera idade e sexo dos metadados oficiais da SVD, calcula o AVQI como índice acústico de gravidade, mede o desempenho por estrato de gravidade, controla idade e sexo e gera o pacote cego para avaliação perceptiva.

Tudo é salvo em `svd_data/revisao_gravidade` (tabelas, figuras em 300 dpi e caches). Rodar de novo reaproveita o que já foi calculado.

In [ ]:
#@title 0 · Configuração, instalação e montagem do Drive
import sys, os, subprocess, warnings
warnings.filterwarnings('ignore')
IN_COLAB = 'google.colab' in sys.modules

def _pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=False)

if IN_COLAB:
    _pip('praat-parselmouth', 'nspfile', 'sbvoicedb', 'xgboost', 'openpyxl')
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as e:
        if 'credential propagation' not in str(e):
            raise
        try:
            drive.mount('/content/drive', force_remount=True)   # segunda tentativa, já com a janela de permissão
        except Exception:
            raise RuntimeError(
                'O Colab não recebeu a permissão do Google Drive. Faça assim: '
                '(1) Ambiente de execução > Desconectar e excluir ambiente de execução; '
                '(2) rode este bloco de novo e, na janela que abrir, escolha a MESMA conta do Drive onde está svd_data '
                'e marque TODAS as permissões (inclusive "Ver, editar, criar e excluir arquivos do Google Drive"); '
                '(3) se a janela não abrir, libere pop-ups e cookies de terceiros para colab.research.google.com.') from None
    DRIVE_ROOT = '/content/drive/MyDrive'
else:
    DRIVE_ROOT = os.environ.get('SVD_TEST_DRIVE', './drive')

# ---------------- caminhos ----------------
SVD_DIR   = os.path.join(DRIVE_ROOT, 'svd_data')            # onde está o features_v5.pkl
OUT_DIR   = os.path.join(SVD_DIR, 'revisao_gravidade')      # tudo que este notebook gera
FIG_DIR   = os.path.join(OUT_DIR, 'figuras')
TAB_DIR   = os.path.join(OUT_DIR, 'tabelas')
CACHE_DIR = os.path.join(OUT_DIR, 'cache')
WAV_TMP   = os.path.join(CACHE_DIR, 'wav_nativo')           # NSP -> WAV na taxa original (50 kHz)
SBV_DIR   = os.path.join(SVD_DIR, 'sbvoicedb')              # usado só se faltar áudio de frase
RATE_DIR  = os.path.join(OUT_DIR, 'avaliacao_perceptiva')
for d in (OUT_DIR, FIG_DIR, TAB_DIR, CACHE_DIR, WAV_TMP, RATE_DIR):
    os.makedirs(d, exist_ok=True)

# ---------------- parâmetros (mesmos do artigo) ----------------
SEED      = 42
N_OUTER   = 5
N_INNER   = 3
N_BOOT    = 2000
N_JOBS    = -1
PAPER_AUC = 0.788
XGB_GRID  = {'clf__n_estimators': [300, 600],          # grade do svd_JoV_revisao_v5
             'clf__max_depth': [3, 4, 6],
             'clf__learning_rate': [0.03, 0.05, 0.10]}
FORCE_RERUN = False          # True refaz CV e AVQI mesmo com cache

# Sessões que aparecem em duas pastas: diagnóstico mantido (primeiro da lista que a sessão tiver)
DUP_DIAG_PRIORITY = ['vocal_fold_paralysis', 'polyp', 'nodule', 'laryngitis', 'hyperfunctional_dysphonia']
CACHE_TAG = '_v4'             # caches de CV com a configuração do XGBoost do v5 (não sobrescreve os da v3)

# Gravidade
HEALTHY_PCTL      = 95        # limiar interno: percentil dos controles saudáveis da própria SVD
AVQI_LIT_CUTOFFS  = [2.43, 2.735, 3.25]   # análise de sensibilidade (faixa da literatura)

# Pacote de avaliação perceptiva
N_RATE_PATH    = 200          # vozes patológicas
N_RATE_HEALTHY = 50           # saudáveis misturadas, para cegamento
RATE_DUP_FRAC  = 0.10         # repetições para confiabilidade intra-avaliador

# Pequenos ajustes só para o teste local fora do Colab (não afeta o Colab)
if not IN_COLAB and os.environ.get('SVD_TEST_FAST'):
    XGB_GRID = {'clf__n_estimators': [50], 'clf__max_depth': [3], 'clf__learning_rate': [0.1]}
    N_BOOT = 100

print('Drive:', DRIVE_ROOT, '| saída:', OUT_DIR)


In [ ]:
#@title 1 · Identidade visual (painéis HTML escuro + laranja, figuras para o artigo)
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, HTML

OR, OR2 = '#FF7A1A', '#FFB070'
BG, CARD, TXT, MUT, OK, BAD = '#111111', '#1C1C1C', '#EDEDED', '#9A9A9A', '#4CC38A', '#FF5C5C'

_CSS = f"""
<style>
.sv-wrap{{background:{BG};color:{TXT};font-family:Inter,Segoe UI,Arial,sans-serif;
  border-radius:10px;padding:18px 22px;margin:10px 0;border-left:5px solid {OR};}}
.sv-wrap h3{{color:{OR};margin:0 0 8px 0;font-size:17px;letter-spacing:.3px}}
.sv-wrap p,.sv-wrap li{{color:{TXT};font-size:13.5px;line-height:1.5}}
.sv-mut{{color:{MUT};font-size:12px}}
.sv-kpis{{display:flex;flex-wrap:wrap;gap:10px;margin:8px 0 4px}}
.sv-kpi{{background:{CARD};border-radius:8px;padding:10px 14px;min-width:130px}}
.sv-kpi b{{display:block;color:{OR};font-size:22px}}
.sv-kpi span{{color:{MUT};font-size:11.5px}}
.sv-tab{{border-collapse:collapse;font-size:12.5px;margin-top:6px;width:100%}}
.sv-tab th{{background:{CARD};color:{OR};text-align:left;padding:6px 9px;border-bottom:1px solid #333}}
.sv-tab td{{padding:5px 9px;border-bottom:1px solid #262626;color:{TXT}}}
.sv-tab tr:hover td{{background:#181818}}
.sv-alert{{border-left-color:{BAD}}} .sv-alert h3{{color:{BAD}}}
.sv-good{{border-left-color:{OK}}} .sv-good h3{{color:{OK}}}
</style>"""
display(HTML(_CSS))

def panel(title, body='', kpis=None, kind='info', note=None):
    cls = {'info': '', 'alert': ' sv-alert', 'good': ' sv-good'}[kind]
    k = ''
    if kpis:
        k = '<div class="sv-kpis">' + ''.join(
            f'<div class="sv-kpi"><b>{v}</b><span>{lab}</span></div>' for lab, v in kpis) + '</div>'
    n = f'<p class="sv-mut">{note}</p>' if note else ''
    display(HTML(_CSS + f'<div class="sv-wrap{cls}"><h3>{title}</h3>{k}{body}{n}</div>'))

def table(df, title, note=None, index=False, kind='info'):
    html = df.to_html(index=index, classes='sv-tab', border=0, escape=False,
                      float_format=lambda x: f'{x:,.3f}')
    panel(title, html, kind=kind, note=note)

def _theme(ax, dark):
    fg = TXT if dark else '#222222'
    ax.set_facecolor(BG if dark else 'white')
    for s in ax.spines.values(): s.set_color('#444' if dark else '#888')
    ax.tick_params(colors=fg); ax.xaxis.label.set_color(fg); ax.yaxis.label.set_color(fg)
    ax.title.set_color(OR if dark else '#111')

def figure(draw, name, figsize=(8, 4.5)):
    """Desenha duas vezes: versão escura para o notebook e versão branca (300 dpi) para o artigo."""
    for dark in (True, False):
        fig, ax = plt.subplots(figsize=figsize, facecolor=BG if dark else 'white')
        draw(fig, ax, dark)
        for a in fig.axes: _theme(a, dark)
        fig.tight_layout()
        if dark:
            plt.show()
        else:
            fig.savefig(os.path.join(FIG_DIR, name + '.png'), dpi=300, facecolor='white')
            plt.close(fig)

def fmt_ci(est, lo, hi, d=3):
    return f'{est:.{d}f} [{lo:.{d}f}; {hi:.{d}f}]'

panel('Revisão JVOICE-D-26-01691 · gravidade, idade e sexo',
      '<p>Este notebook responde aos pontos centrais do Revisor 1: gravidade da disfonia, '
      'idade como confundidor e sexo real em vez do proxy de f0. Todas as tabelas e figuras '
      'ficam salvas em <code>svd_data/revisao_gravidade</code>.</p>')


In [ ]:
#@title 2 · Carrega a matriz de features do artigo
import glob, pickle, re

META_COLS = ['speaker_id', 'session_id', 'label', 'diagnosis', 'vowel', 'pitch', 'wav']

def _as_df(obj):
    if isinstance(obj, pd.DataFrame):
        return obj
    if isinstance(obj, dict):
        dfs = {k: v for k, v in obj.items() if isinstance(v, pd.DataFrame)}
        meta = next((v for k, v in dfs.items() if 'meta' in str(k).lower()), None)
        X = next((v for k, v in dfs.items() if str(k).lower() in ('x', 'features', 'feat', 'x_df', 'df_feat')), None)
        if meta is not None and X is not None and len(meta) == len(X):
            return pd.concat([meta.reset_index(drop=True), X.reset_index(drop=True)], axis=1)
        withlab = [v for v in dfs.values() if 'label' in v.columns]
        if withlab:
            return max(withlab, key=len)
    raise ValueError('formato não reconhecido')

def _load(path):
    if path.endswith('.parquet'):
        return pd.read_parquet(path)
    try:
        return _as_df(pd.read_pickle(path))
    except Exception:
        with open(path, 'rb') as f:
            return _as_df(pickle.load(f))

cands = [p for p in glob.glob(os.path.join(SVD_DIR, '**', '*.pkl'), recursive=True)
         + glob.glob(os.path.join(SVD_DIR, '**', '*.parquet'), recursive=True)
         if OUT_DIR not in p and 'sbvoicedb' not in p and os.path.getsize(p) < 2e9]
cands = sorted(cands, key=lambda p: ('feat' not in os.path.basename(p).lower(), -os.path.getmtime(p)))

loaded = []
for p in cands[:15]:
    try:
        d = _load(p)
        if 'label' in d.columns:
            loaded.append((p, d))
    except Exception:
        pass
if not loaded:
    raise FileNotFoundError(f'Nenhuma matriz com coluna "label" encontrada em {SVD_DIR}')

# prioridade: tamanho idêntico ao artigo (15.766); depois features_v5; depois a maior
pick = next(((p, d) for p, d in loaded if len(d) == 15766), None) \
    or next(((p, d) for p, d in loaded if 'features_v5' in p), None) \
    or max(loaded, key=lambda t: len(t[1]))
FEAT_PATH, df = pick
df = df.reset_index(drop=True).copy()

if 'vowel' in df.columns:
    n_iau = df['vowel'].astype(str).str.lower().str.contains('iau').sum()
    df = df[~df['vowel'].astype(str).str.lower().str.contains('iau')].reset_index(drop=True)
else:
    n_iau = 0

meta_present = [c for c in META_COLS if c in df.columns]
FEAT_COLS = [c for c in df.columns if c not in META_COLS and pd.api.types.is_numeric_dtype(df[c])]
df['label'] = df['label'].astype(int)

panel('Matriz de features carregada',
      f'<p>Arquivo: <code>{FEAT_PATH}</code></p>'
      f'<p class="sv-mut">Colunas de metadados: {", ".join(meta_present)}</p>',
      kpis=[('gravações', f'{len(df):,}'), ('features', len(FEAT_COLS)),
            ('saudáveis (rec.)', f'{(df.label == 0).sum():,}'),
            ('patológicas (rec.)', f'{(df.label == 1).sum():,}'),
            ('/iau/ removidas', n_iau)],
      kind='good' if len(df) == 15766 else 'info',
      note=None if len(df) == 15766 else
      f'O artigo usa 15.766 gravações; esta matriz tem {len(df):,}. A análise segue com ela.')


## Idade, sexo e identidade real do falante

O `summary.csv` oficial da SVD traz, por sessão, a data de nascimento, a data da gravação, o sexo e o ID do falante. Com ele a idade vira dado registrado e o proxy de f0 deixa de ser necessário. O mesmo cruzamento mostra se a unidade chamada de *speaker* no artigo é o falante ou a sessão.

In [ ]:
#@title 3 · Recupera idade e sexo dos metadados oficiais da SVD
import sbvoicedb
SUMMARY = os.path.join(os.path.dirname(sbvoicedb.__file__), 'summary.csv')
S = pd.read_csv(SUMMARY).rename(columns={
    'AufnahmeID': 'svd_session', 'AufnahmeTyp': 'svd_type', 'AufnahmeDatum': 'rec_date',
    'Diagnose': 'diag_note', 'SprecherID': 'svd_speaker', 'Geburtsdatum': 'birth',
    'Geschlecht': 'sex', 'Pathologien': 'pathologies'})
S['age'] = (pd.to_datetime(S.rec_date, errors='coerce') - pd.to_datetime(S.birth, errors='coerce')).dt.days / 365.25
S['sex'] = S['sex'].map({'w': 'F', 'm': 'M'})
S_idx = S.set_index('svd_session')

def _ids(series):
    return pd.to_numeric(series.astype(str).str.extract(r'(\d+)')[0], errors='coerce')

# Qual coluna da matriz corresponde ao ID de sessão (AufnahmeID) da SVD?
cand_cols = {c: _ids(df[c]) for c in ('session_id', 'speaker_id') if c in df.columns}
if 'wav' in df.columns:
    cand_cols['wav (nome do arquivo)'] = pd.to_numeric(
        df['wav'].astype(str).map(os.path.basename).str.extract(r'^(\d+)-')[0], errors='coerce')

rows = []
for name, ids in cand_cols.items():
    t = ids.map(S_idx['svd_type'])
    cov = t.notna().mean()
    agree = ((t[t.notna()] == 'p').astype(int).values == df.loc[t.notna(), 'label'].values).mean() if cov else 0
    rows.append({'coluna': name, 'cobertura': cov, 'concordância rótulo': agree, 'score': cov * agree})
match = pd.DataFrame(rows).sort_values('score', ascending=False)
best = match.iloc[0]
if best['score'] < 0.95:
    table(match, 'Não foi possível casar a matriz com os metadados da SVD', kind='alert')
    raise RuntimeError('Nenhuma coluna casa com o AufnahmeID da SVD (score < 0.95).')

df['svd_session'] = cand_cols[best['coluna']].astype('Int64')
for c in ('svd_speaker', 'sex', 'age', 'pathologies', 'svd_type'):
    df[c] = df['svd_session'].map(S_idx[c])

table(match, 'Casamento matriz ↔ metadados oficiais (summary.csv)',
      note=f'Coluna escolhida: <b>{best["coluna"]}</b>. Idade calculada na data da gravação '
           '(data da sessão − data de nascimento). Fonte: summary.csv distribuído com o pacote sbvoicedb.')

# ---------- sessões que aparecem em mais de uma pasta diagnóstica ----------
dup_diag = df.groupby('svd_session')['diagnosis'].nunique() if 'diagnosis' in df else pd.Series(dtype=int)
multi_diag = dup_diag[dup_diag > 1]
key_cols = [c for c in ('svd_session', 'vowel', 'pitch') if c in df.columns]
n_dup_rec = int(df.duplicated(key_cols).sum()) if len(key_cols) == 3 else 0
combos = (df[df.svd_session.isin(multi_diag.index)].groupby('svd_session')['diagnosis']
            .apply(lambda x: ' + '.join(sorted(set(map(str, x))))).value_counts().rename('sessões').reset_index()
            .rename(columns={'diagnosis': 'pastas em que a mesma sessão aparece'}))
panel('Auditoria: a mesma sessão em mais de uma pasta diagnóstica',
      '<p>Uma sessão com dois diagnósticos (por exemplo laringite e pólipo) aparece na pasta de cada um. '
      'Contada uma vez por pasta, ela vira dois "speakers" na Tabela 1, e suas gravações entram duas vezes na matriz.</p>',
      kpis=[('sessões em >1 pasta', len(multi_diag)),
            ('gravações duplicadas na matriz', f'{n_dup_rec:,}'),
            ('sessões distintas', f'{df.svd_session.nunique():,}'),
            ('"speakers" se contados por pasta', f'{df.groupby(["diagnosis", "svd_session"]).ngroups:,}' if 'diagnosis' in df else '—')],
      kind='alert' if len(multi_diag) else 'good',
      note='No v7 as cópias pesam em dobro no treino e no teste. A partir daqui o notebook mantém uma única cópia '
           'por sessão × vogal × pitch, com o diagnóstico escolhido pela ordem de prioridade do bloco 0 '
           '(lesão orgânica antes de disfonia funcional). O rótulo binário não muda.')
if len(combos):
    table(combos, 'Combinações de pastas das sessões duplicadas')
    combos.to_csv(os.path.join(TAB_DIR, 'sessoes_em_varias_pastas.csv'), index=False)


# ---------- remove as cópias: uma gravação por sessão × vogal × pitch ----------
N_BEFORE_DEDUP = len(df)
if len(multi_diag):
    _prio = {d: i for i, d in enumerate(DUP_DIAG_PRIORITY)}
    df['_prio'] = df['diagnosis'].map(_prio).fillna(len(_prio))
    keep_diag = df[df.svd_session.isin(multi_diag.index)].groupby('svd_session')['_prio'].min()
    drop = df.svd_session.isin(multi_diag.index) & (df['_prio'] != df.svd_session.map(keep_diag))
    df = df[~drop].drop(columns='_prio').reset_index(drop=True)
    df = df.drop_duplicates(key_cols).reset_index(drop=True) if len(key_cols) == 3 else df
    panel('Matriz sem duplicatas', kind='good',
          kpis=[('gravações antes', f'{N_BEFORE_DEDUP:,}'), ('gravações depois', f'{len(df):,}'),
                ('sessões', f'{df.svd_session.nunique():,}')])
else:
    df = df.drop(columns='_prio', errors='ignore')

# ---------------- auditoria: sessão x falante ----------------
sess = (df.groupby('svd_session')
          .agg(label=('label', 'first'), diagnosis=('diagnosis', 'first') if 'diagnosis' in df else ('label', 'first'),
               svd_speaker=('svd_speaker', 'first'), sex=('sex', 'first'), age=('age', 'first'),
               n_rec=('label', 'size'))
          .reset_index())
spk = sess.groupby('svd_speaker').agg(n_sess=('svd_session', 'size'), n_lab=('label', 'nunique'))
multi = spk[spk.n_sess > 1]
both = spk[spk.n_lab > 1]
rec_multi = df['svd_speaker'].isin(multi.index).mean()

# Contaminação esperada se o CV agrupou por sessão (como no v7)
from sklearn.model_selection import StratifiedGroupKFold
contam = []
for seed in range(10):
    sgk = StratifiedGroupKFold(N_OUTER, shuffle=True, random_state=seed)
    for tr, te in sgk.split(df, df['label'], groups=df['svd_session']):
        tr_spk = set(df.loc[tr, 'svd_speaker'])
        contam.append(df.loc[te, 'svd_speaker'].isin(tr_spk).mean())
contam = np.array(contam)

panel('Auditoria: a unidade "speaker" do artigo é a sessão de gravação',
      f'<p>O artigo agrupa por pasta de gravação. Na SVD essa pasta é a <i>sessão</i>, e um mesmo '
      f'falante pode ter várias sessões, inclusive uma saudável e outra patológica. '
      f'Se o v7 agrupou o CV por essa pasta, parte dos falantes de teste também esteve no treino.</p>',
      kpis=[('sessões ("speakers" do artigo)', f'{len(sess):,}'),
            ('falantes reais', f'{sess.svd_speaker.nunique():,}'),
            ('falantes com >1 sessão', f'{len(multi):,}'),
            ('falantes nas duas classes', f'{len(both):,}'),
            ('gravações desses falantes', f'{rec_multi:.1%}'),
            ('teste com falante visto no treino', f'{contam.mean():.1%}')],
      kind='alert' if len(multi) else 'good',
      note='Contaminação estimada em 10 sementes × 5 folds de StratifiedGroupKFold agrupado por sessão. '
           'O bloco 5 refaz o CV agrupando pelo falante real e mede quanto isso muda a AUC.')


by_diag = (sess.groupby('diagnosis')
             .agg(sessões=('svd_session', 'size'), falantes=('svd_speaker', 'nunique'))
             .reset_index())
table(by_diag, 'Sessões x falantes reais por grupo diagnóstico',
      note='Compare com a coluna "Speakers" da Tabela 1 do manuscrito.')
by_diag.to_csv(os.path.join(TAB_DIR, 'sessoes_vs_falantes.csv'), index=False)


In [ ]:
#@title 4 · Idade e sexo por grupo (o que o revisor pede para a Seção 2.2.2)
from scipy import stats
from sklearn.metrics import roc_auc_score

def _desc(g):
    a = g['age']
    return pd.Series({
        'sessões': len(g), 'falantes': g['svd_speaker'].nunique(),
        'mulheres (%)': 100 * (g['sex'] == 'F').mean() if 'sex' in g else np.nan,
        'idade média': a.mean(), 'DP': a.std(),
        'mediana': a.median(), 'IQR': a.quantile(.75) - a.quantile(.25),
        'mín': a.min(), 'máx': a.max()})

sess['grupo'] = np.where(sess.label == 0, 'Saudável', 'Patológico')
demo = pd.concat([
    sess.groupby('grupo').apply(_desc),
    sess[sess.label == 1].groupby('diagnosis').apply(_desc).rename(index=lambda s: '  ' + str(s))])
demo_sex = sess.assign(sexo=sess['sex']).groupby(['grupo', 'sexo']).apply(_desc)[['sessões', 'idade média', 'DP', 'mediana', 'mín', 'máx']]

u = stats.mannwhitneyu(sess.loc[sess.label == 0, 'age'].dropna(), sess.loc[sess.label == 1, 'age'].dropna())
ct = pd.crosstab(sess.grupo, sess.sex)
chi = stats.chi2_contingency(ct)
auc_age = roc_auc_score(sess.label[sess.age.notna()], sess.age.dropna())

table(demo.round(1).reset_index().rename(columns={'index': 'grupo'}),
      'Tabela nova: demografia por grupo (unidade = sessão, idade na data da gravação)')
table(demo_sex.round(1).reset_index(), 'Idade por grupo e sexo')
panel('Idade e sexo como possíveis confundidores',
      kpis=[('Mann–Whitney idade p', f'{u.pvalue:.1e}'),
            ('χ² sexo p', f'{chi[1]:.3f}'),
            ('AUC da idade sozinha', f'{auc_age:.3f}')],
      body='<p>A AUC da idade sozinha mostra quanto do rótulo a idade carrega. '
           'Se ela se aproximar da AUC acústica, o controle de idade (bloco 9) é indispensável.</p>')
demo.round(2).to_csv(os.path.join(TAB_DIR, 'demografia_grupos.csv'))
demo_sex.round(2).to_csv(os.path.join(TAB_DIR, 'demografia_grupo_sexo.csv'))


## Probabilidades fora da amostra

As análises por gravidade precisam da predição de cada sessão. O XGBoost é refeito com o mesmo desenho do artigo (5 folds externos, 3 internos), uma vez agrupando por sessão e outra agrupando pelo falante real. A segunda é a usada daqui em diante.

In [ ]:
#@title 5 · XGBoost com CV aninhado: agrupando por sessão (como no artigo) e por falante real
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import roc_curve

def _device():
    try:
        import torch
        return 'cuda' if torch.cuda.is_available() else 'cpu'
    except Exception:
        return 'cpu'
DEVICE = _device()

X = df[FEAT_COLS].values.astype(np.float32)
y = df['label'].values

def nested_oof(groups, tag):
    cache = os.path.join(CACHE_DIR, f'oof_xgb_{tag}{CACHE_TAG}.parquet')
    if os.path.exists(cache) and not FORCE_RERUN:
        o = pd.read_parquet(cache)
        if len(o) == len(df):
            return o['prob'].values, o['fold'].values
    prob = np.full(len(df), np.nan); fold = np.full(len(df), -1)
    outer = StratifiedGroupKFold(N_OUTER, shuffle=True, random_state=SEED)
    for k, (tr, te) in enumerate(outer.split(X, y, groups)):
        assert not set(groups[tr]) & set(groups[te])
        inner = StratifiedGroupKFold(N_INNER, shuffle=True, random_state=SEED)
        inner_splits = list(inner.split(X[tr], y[tr], groups[tr]))
        pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                         ('clf', XGBClassifier(tree_method='hist', device=DEVICE, eval_metric='logloss', verbosity=0,
                                               scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
                                               random_state=SEED, n_jobs=-1 if DEVICE == 'cpu' else 1))])
        gs = GridSearchCV(pipe, XGB_GRID, scoring='roc_auc', cv=inner_splits, n_jobs=1, refit=True)
        gs.fit(X[tr], y[tr])
        prob[te] = gs.predict_proba(X[te])[:, 1]; fold[te] = k
        print(f'[{tag}] fold {k + 1}/{N_OUTER}  AUC={roc_auc_score(y[te], prob[te]):.3f}  {gs.best_params_}')
    pd.DataFrame({'prob': prob, 'fold': fold}).to_parquet(cache)
    return prob, fold

def boot_auc(prob, mask=None, n=N_BOOT, unit='svd_speaker'):
    """AUC com IC por bootstrap de falantes reais (todas as gravações do falante entram juntas)."""
    d = pd.DataFrame({'y': y, 'p': prob, 'u': df[unit].values})
    if mask is not None:
        d = d[mask]
    units = d['u'].unique(); g = {k: v.index.values for k, v in d.groupby('u')}
    rng = np.random.default_rng(SEED); est = roc_auc_score(d.y, d.p); bs = []
    for _ in range(n):
        idx = np.concatenate([g[u] for u in rng.choice(units, len(units))])
        yy = d.loc[idx, 'y']
        if yy.nunique() == 2:
            bs.append(roc_auc_score(yy, d.loc[idx, 'p']))
    return est, *np.percentile(bs, [2.5, 97.5])

prob_sess, _ = nested_oof(df['svd_session'].astype(str).values, 'grupo_sessao')
prob_spk, fold_spk = nested_oof(df['svd_speaker'].astype(str).values, 'grupo_falante')
df['prob'] = prob_spk; df['fold'] = fold_spk

a_s = boot_auc(prob_sess); a_p = boot_auc(prob_spk)
fpr, tpr, thr = roc_curve(y, prob_spk); j = np.argmax(tpr - fpr)
THR_REC = thr[j]; SE_REC, SP_REC = tpr[j], 1 - fpr[j]

# nível de sessão: média das probabilidades das ~12 gravações
sess = sess.merge(df.groupby('svd_session')['prob'].mean().rename('prob_sess').reset_index(), on='svd_session')
fpr_s, tpr_s, thr_s = roc_curve(sess.label, sess.prob_sess); js = np.argmax(tpr_s - fpr_s)
THR_SESS = thr_s[js]
sess['pred'] = (sess.prob_sess >= THR_SESS).astype(int)

panel('Desempenho com separação por falante real',
      kpis=[('AUC agrupando por sessão', fmt_ci(*a_s)), ('AUC agrupando por falante', fmt_ci(*a_p)),
            ('Δ AUC', f'{a_p[0] - a_s[0]:+.3f}'),
            ('Se / Sp (gravação)', f'{SE_REC:.3f} / {SP_REC:.3f}'),
            ('Se / Sp (sessão)', f'{tpr_s[js]:.3f} / {1 - fpr_s[js]:.3f}')],
      body=f'<p>AUC do artigo: {PAPER_AUC}. O agrupamento por sessão deve reproduzi-la; '
           f'a diferença para o agrupamento por falante mede o efeito do vazamento.</p>',
      note=f'IC 95% por bootstrap de falantes reais ({N_BOOT} reamostragens). '
           f'Limiar de Youden: {THR_REC:.3f} (gravação), {THR_SESS:.3f} (sessão). Dispositivo: {DEVICE}.')
pd.DataFrame([{'agrupamento': 'sessão', 'AUC': a_s[0], 'IC_inf': a_s[1], 'IC_sup': a_s[2]},
              {'agrupamento': 'falante real', 'AUC': a_p[0], 'IC_inf': a_p[1], 'IC_sup': a_p[2]}]
             ).to_csv(os.path.join(TAB_DIR, 'auc_sessao_vs_falante.csv'), index=False)


## Gravidade acústica

A SVD não tem avaliação perceptiva. O AVQI v03.01 é o índice acústico de gravidade com validação contra a escala G, e a SVD tem o que ele exige: vogal sustentada e fala encadeada (*Guten Morgen, wie geht es Ihnen?*). O script original é executado sem alterações.

In [ ]:
#@title 6 · Gravidade acústica: AVQI v03.01 (vogal /a/ + frase) na taxa original de 50 kHz
import soundfile as sf
from joblib import Parallel, delayed

# Script original do AVQI v03.01 (Maryn, Corthals, Barsties), partes 0–2, sem nenhuma alteração.
# O CPPS da frase sozinha (cpps_cs), segundo indicador de gravidade que não usa a vogal sustentada,
# é calculado à parte, com os mesmos parâmetros de cepstrograma do AVQI.
AVQI_SCRIPT = r'''
form AVQI
    sentence sv_path
    sentence cs_path
endform

Read from file: sv_path$
Rename: "sv"
Read from file: cs_path$
Rename: "cs"

# --------------------------------------------------------------------------------------------
# PART 0:
# HIGH-PASS FILTERING OF THE SOUND FILES.
# --------------------------------------------------------------------------------------------

select Sound cs
Filter (stop Hann band)... 0 34 0.1
Rename... cs2
select Sound sv
Filter (stop Hann band)... 0 34 0.1
Rename... sv2

# --------------------------------------------------------------------------------------------
# PART 1:
# DETECTION, EXTRACTION AND CONCATENATION OF
# THE VOICED SEGMENTS IN THE RECORDING
# OF CONTINUOUS SPEECH.
# --------------------------------------------------------------------------------------------

select Sound cs2
Copy... original
samplingRate = Get sampling frequency
intermediateSamples = Get sampling period
Create Sound... onlyVoice 0 0.001 'samplingRate' 0
select Sound original
To TextGrid (silences)... 50 0.003 -25 0.1 0.1 silence sounding
select Sound original
plus TextGrid original
Extract intervals where... 1 no "does not contain" silence
Concatenate
select Sound chain
Rename... onlyLoud
globalPower = Get power in air
select TextGrid original
Remove

select Sound onlyLoud
signalEnd = Get end time
windowBorderLeft = Get start time
windowWidth = 0.03
windowBorderRight = windowBorderLeft + windowWidth
globalPower = Get power in air
voicelessThreshold = globalPower*(30/100)

select Sound onlyLoud
extremeRight = signalEnd - windowWidth
while windowBorderRight < extremeRight
	Extract part... 'windowBorderLeft' 'windowBorderRight' Rectangular 1.0 no
	select Sound onlyLoud_part
	partialPower = Get power in air
	if partialPower > voicelessThreshold
		call checkZeros 0
		if (zeroCrossingRate <> undefined) and (zeroCrossingRate < 3000)
			select Sound onlyVoice
			plus Sound onlyLoud_part
			Concatenate
			Rename... onlyVoiceNew
			select Sound onlyVoice
			Remove
			select Sound onlyVoiceNew
			Rename... onlyVoice
		endif
	endif
	select Sound onlyLoud_part
	Remove
	windowBorderLeft = windowBorderLeft + 0.03
	windowBorderRight = windowBorderLeft + 0.03
	select Sound onlyLoud
endwhile
select Sound onlyVoice

procedure checkZeros zeroCrossingRate

	start = 0.0025
	startZero = Get nearest zero crossing... 'start'
	findStart = startZero
	findStartZeroPlusOne = startZero + intermediateSamples
	startZeroPlusOne = Get nearest zero crossing... 'findStartZeroPlusOne'
	zeroCrossings = 0
	strips = 0

	while (findStart < 0.0275) and (findStart <> undefined)
		while startZeroPlusOne = findStart
			findStartZeroPlusOne = findStartZeroPlusOne + intermediateSamples
			startZeroPlusOne = Get nearest zero crossing... 'findStartZeroPlusOne'
		endwhile
		afstand = startZeroPlusOne - startZero
		strips = strips +1
		zeroCrossings = zeroCrossings +1
		findStart = startZeroPlusOne
	endwhile
	zeroCrossingRate = zeroCrossings/afstand
endproc

# --------------------------------------------------------------------------------------------
# PART 2:
# DETERMINATION OF THE SIX ACOUSTIC MEASURES
# AND CALCULATION OF THE ACOUSTIC VOICE QUALITY INDEX.
# --------------------------------------------------------------------------------------------

select Sound sv2
durationVowel = Get total duration
durationStart=durationVowel-3
if durationVowel>3
Extract part... durationStart durationVowel rectangular 1 no
Rename... sv3
elsif durationVowel<=3
Copy... sv3
endif

select Sound onlyVoice
durationOnlyVoice = Get total duration
plus Sound sv3
Concatenate
Rename... avqi
durationAll = Get total duration
minimumSPL = Get minimum... 0 0 None
maximumSPL = Get maximum... 0 0 None

# Narrow-band spectrogram and LTAS

To Spectrogram... 0.03 4000 0.002 20 Gaussian
select Sound avqi
To Ltas... 1
minimumSpectrum = Get minimum... 0 4000 None
maximumSpectrum = Get maximum... 0 4000 None

# Power-cepstrogram, Cepstral peak prominence and Smoothed cepstral peak prominence

select Sound avqi
To PowerCepstrogram... 60 0.002 5000 50
cpps = Get CPPS... no 0.01 0.001 60 330 0.05 Parabolic 0.001 0 Straight Robust
To PowerCepstrum (slice)... 0.1
maximumCepstrum = Get peak... 60 330 None

# Slope of the long-term average spectrum

select Sound avqi
To Ltas... 1
slope = Get slope... 0 1000 1000 10000 energy

# Tilt of trendline through the long-term average spectrum

select Ltas avqi
Compute trend line... 1 10000
tilt = Get slope... 0 1000 1000 10000 energy

# Amplitude perturbation measures

select Sound avqi
To PointProcess (periodic, cc)... 50 400
Rename... avqi1
select Sound avqi
plus PointProcess avqi1
percentShimmer = Get shimmer (local)... 0 0 0.0001 0.02 1.3 1.6
shim = percentShimmer*100
shdb = Get shimmer (local_dB)... 0 0 0.0001 0.02 1.3 1.6

# Harmonic-to-noise ratio

select Sound avqi
To Pitch (cc)... 0 75 15 no 0.03 0.45 0.01 0.35 0.14 600
select Sound avqi
plus Pitch avqi
To PointProcess (cc)
Rename... avqi2
select Sound avqi
plus Pitch avqi
plus PointProcess avqi2
voiceReport$ = Voice report... 0 0 75 600 1.3 1.6 0.03 0.45
hnr = extractNumber (voiceReport$, "Mean harmonics-to-noise ratio: ")

# Calculation of the AVQI

avqi = (4.152-(0.177*cpps)-(0.006*hnr)-(0.037*shim)+(0.941*shdb)+(0.01*slope)+(0.093*tilt))*2.8902
'''

# ---------- 6a. localiza os NSP originais de /a/ normal e da frase ----------
def _index_audio(root):
    out = {}
    for p in glob.iglob(os.path.join(root, '**', '*'), recursive=True):
        b = os.path.basename(p).lower()
        m = re.match(r'^(\d+)-(a_n|phrase)\.(nsp|wav)$', b)
        if m and 'egg' not in b and OUT_DIR not in p:
            key = (int(m[1]), m[2]); ext = m[3]
            if key not in out or (ext == 'nsp' and not out[key].lower().endswith('.nsp')):
                out[key] = p
    return out

need = set(sess.svd_session.astype(int))
AUD = _index_audio(SVD_DIR)

def _native_ok(p):
    if p.lower().endswith('.nsp'):
        return True
    try:
        return sf.info(p).samplerate >= 44100
    except Exception:
        return False

def _coverage(aud):
    sv = sum((s, 'a_n') in aud and _native_ok(aud[(s, 'a_n')]) for s in need)
    cs = sum((s, 'phrase') in aud and _native_ok(aud[(s, 'phrase')]) for s in need)
    return sv / len(need), cs / len(need)

cov_sv, cov_cs = _coverage(AUD)
src_note = 'arquivos encontrados no Drive'
if min(cov_sv, cov_cs) < 0.95:
    # Fallback: baixa do Zenodo (CC BY 4.0) só as patologias do estudo + saudáveis, só a_n e frase
    try:
        from sbvoicedb import SbVoiceDb, Recording, Pathology
        PATHOS = ['Hyperfunktionelle Dysphonie', 'Rekurrensparese', 'Laryngitis',
                  'Stimmlippenpolyp', 'Phonationsknötchen']
        db = SbVoiceDb(SBV_DIR, include_healthy=True,
                       pathology_filter=Pathology.name.in_(PATHOS),
                       recording_filter=Recording.utterance.in_(['a_n', 'phrase']),
                       download_mode='lazy')
        for rec in db.iter_recordings():
            f = rec.nspfile if os.path.isabs(rec.nspfile) else os.path.join(db.datadir, rec.nspfile)
            AUD.setdefault((int(rec.session_id), rec.utterance), f)
        cov_sv, cov_cs = _coverage(AUD)
        src_note = 'Drive + download do Zenodo via sbvoicedb'
    except Exception as e:
        panel('Download complementar falhou', f'<p><code>{str(e)[:400]}</code></p>', kind='alert',
              note='O AVQI segue com as sessões que têm vogal e frase na taxa original.')

panel('Áudio para o AVQI', kpis=[('/a/ normal (≥44,1 kHz)', f'{cov_sv:.1%}'),
                                ('frase (≥44,1 kHz)', f'{cov_cs:.1%}')],
      note=f'Fonte: {src_note}. Os WAV de 16 kHz do artigo não servem para o AVQI, porque a '
           'inclinação e o tilt do LTAS vão até 10 kHz; por isso o cálculo usa os NSP originais (50 kHz).')

# ---------- 6b. cálculo por sessão (worker em módulo próprio para rodar em paralelo) ----------
WORKER_SRC = r'''
import os, re, numpy as np, soundfile as sf, nspfile, parselmouth
from parselmouth.praat import call

def to_wav(p, tag, wav_dir):
    if not p.lower().endswith('.nsp'):
        return p
    out = os.path.join(wav_dir, tag + '.wav')
    if not os.path.exists(out):
        fs, x = nspfile.read(p)
        x = np.asarray(x)
        if x.ndim > 1:
            x = x[:, 0]
        sf.write(out, (x / 32768.0).astype(np.float32), fs, subtype='FLOAT')
    return out

def voice_report(wav):
    snd = parselmouth.Sound(wav)
    pitch = call(snd, 'To Pitch (cc)', 0, 75, 15, 'no', 0.03, 0.45, 0.01, 0.35, 0.14, 600)
    pp = call([snd, pitch], 'To PointProcess (cc)')
    rep = call([snd, pitch, pp], 'Voice report', 0, 0, 75, 600, 1.3, 1.6, 0.03, 0.45)
    def num(lbl):
        m = re.search(re.escape(lbl) + r'\s*([-\d.]+)', rep)
        return float(m.group(1)) if m else np.nan
    f0 = pitch.selected_array['frequency']; f0 = f0[f0 > 0]
    ratio = f0[1:] / f0[:-1] if len(f0) > 2 else np.array([1.0])
    octave = float(np.mean((np.abs(ratio - 0.5) < 0.06) | (np.abs(ratio - 2.0) < 0.2)))
    return {'unvoiced_pct': num('Fraction of locally unvoiced frames:'),
            'n_breaks': num('Number of voice breaks:'),
            'breaks_pct': num('Degree of voice breaks:'),
            'octave_jump_frac': octave, 'sv_dur': snd.get_total_duration()}

def cpps_phrase(wav):
    snd = call(parselmouth.Sound(wav), 'Filter (stop Hann band)', 0, 34, 0.1)
    pc = call(snd, 'To PowerCepstrogram', 60, 0.002, 5000, 50)
    return float(call(pc, 'Get CPPS', 'no', 0.01, 0.001, 60, 330, 0.05, 'Parabolic', 0.001, 0, 'Straight', 'Robust'))

def run(s, sv_src, cs_src, wav_dir, script):
    r = {'svd_session': s}
    sv = to_wav(sv_src, f'{s}-a_n', wav_dir); cs = to_wav(cs_src, f'{s}-phrase', wav_dir)
    try:
        v = parselmouth.praat.run(script, sv, cs, return_variables=True)[-1]
        r.update({k: float(v[k]) for k in ('avqi', 'cpps', 'hnr', 'shim', 'shdb', 'slope', 'tilt')})
        r['cs_voiced_dur'] = float(v['durationOnlyVoice'])
    except Exception as e:
        r['error'] = str(e)[:200]
    for fn in (lambda: r.update(voice_report(sv)), lambda: r.update({'cpps_cs': cpps_phrase(cs)})):
        try:
            fn()
        except Exception:
            pass
    return r
'''
open(os.path.join(CACHE_DIR, 'svd_avqi_worker.py'), 'w').write(WORKER_SRC)
sys.path.insert(0, CACHE_DIR)
os.environ['PYTHONPATH'] = CACHE_DIR + os.pathsep + os.environ.get('PYTHONPATH', '')
import importlib, svd_avqi_worker
importlib.reload(svd_avqi_worker)
_to_wav = lambda p, tag: svd_avqi_worker.to_wav(p, tag, WAV_TMP)

AVQI_CACHE = os.path.join(CACHE_DIR, 'avqi_por_sessao.csv')
done = pd.read_csv(AVQI_CACHE) if os.path.exists(AVQI_CACHE) and not FORCE_RERUN else pd.DataFrame()
todo = [s for s in sorted(need) if (s, 'a_n') in AUD and (s, 'phrase') in AUD
        and _native_ok(AUD[(s, 'a_n')]) and _native_ok(AUD[(s, 'phrase')])
        and (done.empty or s not in set(done.svd_session))]
if todo:
    print(f'Calculando AVQI para {len(todo)} sessões...')
    new = Parallel(n_jobs=N_JOBS, verbose=5)(
        delayed(svd_avqi_worker.run)(s, AUD[(s, 'a_n')], AUD[(s, 'phrase')], WAV_TMP, AVQI_SCRIPT) for s in todo)
    done = pd.concat([done, pd.DataFrame(new)], ignore_index=True)
    done.to_csv(AVQI_CACHE, index=False)

AV = done.copy().rename(columns={c: 'avqi_' + c for c in ('cpps', 'hnr', 'shim', 'shdb', 'slope', 'tilt')})
if 'error' not in AV:
    AV['error'] = np.nan
sess = sess.drop(columns=[c for c in AV.columns if c in sess.columns and c != 'svd_session'])
sess = sess.merge(AV, on='svd_session', how='left')
ok = sess.avqi.notna()
fail = sess[sess.error.notna()]
panel('AVQI calculado', kpis=[('sessões com AVQI', f'{ok.sum():,} / {len(sess):,}'),
                              ('falhas (saudáveis / patológicas)',
                               f'{(fail.label == 0).sum()} / {(fail.label == 1).sum()}'),
                              ('duração média da vogal (s)', f'{sess.sv_dur.mean():.2f}'),
                              ('fala vozeada média (s)', f'{sess.cs_voiced_dur.mean():.2f}')],
      note='Desvio de protocolo a declarar: o AVQI prevê os últimos 3 s da vogal e um trecho de leitura; '
           'a SVD oferece vogais de ~1–2 s e uma única frase curta. O script usa a vogal inteira quando ela tem menos de 3 s. '
           'Falhas em vozes patológicas costumam ser sinais tão aperiódicos que o CPPS fica indefinido; '
           'elas entram como estrato próprio ("AVQI não computável") no bloco 7.')


## Desempenho por estrato de gravidade

Esta é a resposta direta ao revisor: qual a composição de gravidade da amostra e quanto o instrumento detecta em cada nível.

In [ ]:
#@title 7 · Distribuição de gravidade e desempenho do classificador por estrato
H = sess[(sess.label == 0) & sess.avqi.notna()]
P = sess[(sess.label == 1) & sess.avqi.notna()].copy()
AVQI_THR = np.percentile(H.avqi, HEALTHY_PCTL)

# Estratos: S0 = dentro da faixa dos saudáveis; acima dela, tercis da própria amostra patológica
above = P.avqi > AVQI_THR
t1, t2 = np.percentile(P.loc[above, 'avqi'], [100 / 3, 200 / 3]) if above.sum() >= 3 else (np.inf, np.inf)
def _stratum(a):
    if np.isnan(a): return np.nan
    if a <= AVQI_THR: return 'S0 · faixa normal'
    if a <= t1: return 'S1 · leve'
    if a <= t2: return 'S2 · moderada'
    return 'S3 · acentuada'
sess['estrato'] = np.where(sess.label == 0, 'Saudável', sess.avqi.map(_stratum))
NC = 'AVQI não computável'
sess.loc[(sess.label == 1) & sess.avqi.isna() & sess.error.notna(), 'estrato'] = NC
P['estrato'] = P.avqi.map(_stratum)
ORDER = ['S0 · faixa normal', 'S1 · leve', 'S2 · moderada', 'S3 · acentuada']

# ---------- 7a. distribuição por diagnóstico ----------
rows = []
for name, g in [('Saudável', H)] + [(d, P[P.diagnosis == d]) for d in sorted(P.diagnosis.unique())] + [('Todos patológicos', P)]:
    r = {'grupo': name, 'n': len(g), 'AVQI mediana': g.avqi.median(),
         'IQR': g.avqi.quantile(.75) - g.avqi.quantile(.25),
         f'% > P{HEALTHY_PCTL} saudáveis': 100 * (g.avqi > AVQI_THR).mean()}
    for c in AVQI_LIT_CUTOFFS:
        r[f'% > {c}'] = 100 * (g.avqi > c).mean()
    if name != 'Saudável':
        for s in ORDER:
            r[s] = 100 * (g.avqi.map(_stratum) == s).mean()
    rows.append(r)
DIST = pd.DataFrame(rows)
table(DIST.round(1), 'Gravidade acústica por grupo diagnóstico (AVQI v03.01)',
      note=f'Limiar interno: P{HEALTHY_PCTL} dos controles saudáveis da SVD = {AVQI_THR:.2f}. '
           f'Acima dele, tercis da amostra patológica ({t1:.2f} e {t2:.2f}). '
           'As colunas com limiares da literatura servem de análise de sensibilidade.')
DIST.round(3).to_csv(os.path.join(TAB_DIR, 'gravidade_por_diagnostico.csv'), index=False)

def _draw_dist(fig, ax, dark):
    groups = ['Saudável'] + sorted(P.diagnosis.unique())
    data = [H.avqi.values] + [P.loc[P.diagnosis == d, 'avqi'].values for d in groups[1:]]
    bp = ax.boxplot(data, vert=False, patch_artist=True, showfliers=False, widths=.55)
    for i, b in enumerate(bp['boxes']):
        b.set_facecolor('#3A3A3A' if i == 0 else OR); b.set_alpha(.75); b.set_edgecolor(TXT if dark else '#333')
    for k in ('medians', 'whiskers', 'caps'):
        for l in bp[k]: l.set_color(TXT if dark else '#333')
    rng = np.random.default_rng(0)
    for i, v in enumerate(data):
        ax.scatter(v, i + 1 + rng.uniform(-.18, .18, len(v)), s=4, alpha=.25, color=OR2 if dark else '#555')
    ax.axvline(AVQI_THR, ls='--', color=BAD, lw=1.2, label=f'P{HEALTHY_PCTL} saudáveis = {AVQI_THR:.2f}')
    ax.set_yticks(range(1, len(groups) + 1)); ax.set_yticklabels([str(g) for g in groups])
    ax.set_xlabel('AVQI v03.01 (maior = pior qualidade vocal)'); ax.set_title('Distribuição de gravidade acústica')
    leg = ax.legend(loc='lower right', fontsize=8, frameon=False)
    for t in leg.get_texts(): t.set_color(TXT if dark else '#222')
figure(_draw_dist, 'fig_gravidade_distribuicao', (8.5, 4.6))

# ---------- 7b. desempenho por estrato ----------
H_sess = sess[sess.label == 0]
def _auc_boot_sess(sub):
    d = pd.concat([H_sess[['label', 'prob_sess', 'svd_speaker']], sub[['label', 'prob_sess', 'svd_speaker']]])
    d = d.reset_index(drop=True); g = {k: v.index.values for k, v in d.groupby('svd_speaker')}
    units = np.array(list(g)); rng = np.random.default_rng(SEED); bs = []
    est = roc_auc_score(d.label, d.prob_sess)
    se = (sub.prob_sess >= THR_SESS).mean(); bse = []
    for _ in range(N_BOOT):
        idx = np.concatenate([g[u] for u in rng.choice(units, len(units))])
        dd = d.loc[idx]
        if dd.label.nunique() == 2:
            bs.append(roc_auc_score(dd.label, dd.prob_sess))
            bse.append((dd.loc[dd.label == 1, 'prob_sess'] >= THR_SESS).mean())
    return est, *np.percentile(bs, [2.5, 97.5]), se, *np.percentile(bse, [2.5, 97.5])

perf = []
P_s = sess[(sess.label == 1) & sess.avqi.notna()]
for s in ['Todos'] + ORDER + [NC]:
    sub = P_s if s == 'Todos' else (sess[sess.estrato == NC] if s == NC else P_s[P_s.estrato == s])
    if len(sub) < 5:
        continue
    a, al, ah, se, sl, sh = _auc_boot_sess(sub)
    perf.append({'estrato': s, 'sessões': len(sub), 'falantes': sub.svd_speaker.nunique(),
                 'AVQI mediana': sub.avqi.median(),
                 'AUC vs saudáveis': fmt_ci(a, al, ah), 'Sensibilidade': fmt_ci(se, sl, sh),
                 '_auc': a, '_se': se})
sp_h = 1 - (H_sess.prob_sess >= THR_SESS).mean()
PERF = pd.DataFrame(perf)
table(PERF.drop(columns=['_auc', '_se']),
      'Desempenho do XGBoost por estrato de gravidade (nível de sessão)',
      note=f'Cada estrato patológico é comparado com todos os saudáveis (especificidade fixa = {sp_h:.3f}). '
           f'Limiar de Youden da sessão = {THR_SESS:.3f}. IC 95% por bootstrap de falantes reais.')
PERF.to_csv(os.path.join(TAB_DIR, 'desempenho_por_estrato.csv'), index=False)

# ---------- 7c. sensibilidade como função contínua da gravidade ----------
P_s = P_s.assign(det=(P_s.prob_sess >= THR_SESS).astype(int))
P_s['decil'] = pd.qcut(P_s.avqi, 10, labels=False, duplicates='drop')
rng = np.random.default_rng(SEED); curve = []
for q, g in P_s.groupby('decil'):
    bs = [g.det.values[rng.integers(0, len(g), len(g))].mean() for _ in range(N_BOOT)]
    curve.append((g.avqi.median(), g.det.mean(), *np.percentile(bs, [2.5, 97.5])))
curve = np.array(curve)

def _draw_curve(fig, ax, dark):
    ax.fill_between(curve[:, 0], curve[:, 2], curve[:, 3], color=OR, alpha=.22, lw=0)
    ax.plot(curve[:, 0], curve[:, 1], 'o-', color=OR, lw=2, ms=5, label='Sensibilidade (decis de AVQI)')
    ax.axhline(sp_h, color=OK, ls=':', lw=1.4, label=f'Especificidade nos saudáveis = {sp_h:.2f}')
    ax.axvline(AVQI_THR, color=BAD, ls='--', lw=1.1, label=f'P{HEALTHY_PCTL} saudáveis')
    ax.set_ylim(0, 1.02); ax.set_xlabel('AVQI mediano do decil'); ax.set_ylabel('Proporção detectada')
    ax.set_title('Sensibilidade do XGBoost ao longo da gravidade')
    leg = ax.legend(fontsize=8, frameon=False, loc='lower right')
    for t in leg.get_texts(): t.set_color(TXT if dark else '#222')
figure(_draw_curve, 'fig_sensibilidade_vs_gravidade', (8, 4.4))

# ---------- 7d. concordância com um indicador que não usa a vogal: CPPS da frase ----------
rho = stats.spearmanr(P.avqi, P.cpps_cs, nan_policy='omit')
P_s['q_cpps_cs'] = pd.qcut(-P_s.cpps_cs, 4, labels=['Q1 · leve', 'Q2', 'Q3', 'Q4 · acentuada'])
cc = (P_s.groupby('q_cpps_cs', observed=True)
        .agg(sessões=('det', 'size'), CPPS_frase=('cpps_cs', 'median'), sensibilidade=('det', 'mean'))
        .reset_index())
table(cc.round(3), 'Checagem de circularidade: estratos pelo CPPS da frase (fala encadeada)',
      note=f'O AVQI usa CPPS, HNR e shimmer da mesma vogal que alimenta o classificador. O CPPS da frase não '
           f'usa a vogal. Spearman AVQI × CPPS da frase nos patológicos: ρ = {rho.correlation:.2f}. '
           'Se a sensibilidade sobe nos dois esquemas, o padrão não é artefato de circularidade.')
cc.to_csv(os.path.join(TAB_DIR, 'sensibilidade_por_cpps_frase.csv'), index=False)


## Jitter e shimmer em vozes mais graves

O revisor argumenta que perturbação perde confiabilidade com aperiodicidade extrema. Aqui os efeitos são recalculados por estrato e só em sinais Tipo I.

In [ ]:
#@title 8 · Efeito de cada parâmetro Praat por estrato e tipologia de sinal (Titze)
PRAAT_PATTERNS = [
    ('Jitter PPQ5', r'ppq5'), ('Jitter RAP', r'rap'),
    ('Jitter local (%)', r'jitt\w*loc|^jitter(_pct|_percent|%)?$|localjitter'),
    ('Shimmer APQ3', r'apq3'), ('Shimmer APQ5', r'apq5'),
    ('Shimmer (dB)', r'shim\w*db|db\w*shim'), ('Shimmer local', r'shim\w*loc|^shimmer$|localshimmer'),
    ('CPPS (dB)', r'cpp'), ('HNR (dB)', r'hnr'),
    ('f0 SD (Hz)', r'(f0|fo|pitch)\w*(sd|std|stdev|dev)|(sd|std|stdev)\w*(f0|fo|pitch)'),
    ('Mean f0 (Hz)', r'(f0|fo|pitch)\w*(mean|avg|med)|(mean|avg)\w*(f0|fo|pitch)')]
_skip = re.compile(r'mfcc|spec|zcr|rms|delta|centroid|roll|flat|bandw', re.I)
PRAAT = {}
used = set()
for lab, pat in PRAAT_PATTERNS:
    hit = [c for c in FEAT_COLS if re.search(pat, c, re.I) and not _skip.search(c) and c not in used]
    if hit:
        PRAAT[lab] = hit[0]; used.add(hit[0])
missing = [lab for lab, _ in PRAAT_PATTERNS if lab not in PRAAT]
cand_f0 = [c for c in FEAT_COLS if re.search(r'f0|fo_|pitch|freq', c, re.I) and not _skip.search(c)]
table(pd.DataFrame({'parâmetro': list(PRAAT), 'coluna na matriz': list(PRAAT.values())}),
      'Parâmetros Praat identificados na matriz',
      note=(f'Não identificados: {", ".join(missing)}. Colunas candidatas de f0 na matriz: '
            f'{", ".join(cand_f0) or "nenhuma"}.') if missing else None)

# medianas por sessão (f0 SD só no pitch normal, como na Tabela 3)
normal = df['pitch'].astype(str).str.lower().isin(['n', 'normal']) if 'pitch' in df else pd.Series(True, index=df.index)
med = df.groupby('svd_session')[[c for k, c in PRAAT.items() if k != 'f0 SD (Hz)']].median()
if 'f0 SD (Hz)' in PRAAT:
    med[PRAAT['f0 SD (Hz)']] = df[normal].groupby('svd_session')[PRAAT['f0 SD (Hz)']].median()
SM = sess.merge(med.reset_index(), on='svd_session', how='left')

def rbc(h, p):
    h, p = np.asarray(h, float), np.asarray(p, float)
    h, p = h[~np.isnan(h)], p[~np.isnan(p)]
    if len(h) < 3 or len(p) < 3: return np.nan
    U = stats.mannwhitneyu(h, p).statistic
    return 2 * U / (len(h) * len(p)) - 1

H_sm = SM[SM.label == 0]
eff = {}
for s in ['Todos'] + ORDER:
    sub = SM[(SM.label == 1)] if s == 'Todos' else SM[SM.estrato == s]
    eff[s] = {k: rbc(H_sm[c], sub[c]) for k, c in PRAAT.items()}
EFF = pd.DataFrame(eff)
table(EFF.round(3).reset_index().rename(columns={'index': 'parâmetro'}),
      'Correlação rank-bisserial (saudáveis × cada estrato), unidade = sessão',
      note='Negativo = valor maior no grupo patológico (mesma convenção da Tabela 3). '
           'Responde ao revisor: mostra se jitter e shimmer continuam liderando nos estratos mais graves '
           'ou se o CPPS assume a frente quando a aperiodicidade aumenta.')
EFF.round(4).to_csv(os.path.join(TAB_DIR, 'efeito_por_estrato.csv'))

def _draw_heat(fig, ax, dark):
    M = EFF[ORDER].abs().values
    im = ax.imshow(M, cmap='Oranges', vmin=0, vmax=max(.6, np.nanmax(M)), aspect='auto')
    ax.set_xticks(range(len(ORDER))); ax.set_xticklabels(ORDER, rotation=20, ha='right')
    ax.set_yticks(range(len(EFF))); ax.set_yticklabels(EFF.index)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            if not np.isnan(M[i, j]):
                ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=8,
                        color='black' if M[i, j] > .3 else ('white' if dark else '#222'))
    ax.set_title('|r| rank-bisserial por estrato de gravidade')
    cb = fig.colorbar(im, ax=ax, fraction=.04); cb.ax.tick_params(colors=TXT if dark else '#222')
figure(_draw_heat, 'fig_efeito_por_estrato', (8, 5.2))

# ---------- tipologia de sinal (Titze 1995; operacionalização automática) ----------
def _sig_type(r):
    if np.isnan(r.unvoiced_pct): return np.nan
    if r.unvoiced_pct >= 30 or r.breaks_pct >= 20: return 'Tipo III'
    if r.unvoiced_pct < 5 and r.n_breaks == 0 and r.octave_jump_frac < 0.01: return 'Tipo I'
    return 'Tipo II'
SM['tipo_sinal'] = SM.apply(_sig_type, axis=1)
sess['tipo_sinal'] = SM['tipo_sinal'].values
TT = pd.crosstab(SM.estrato, SM.tipo_sinal, normalize='index').mul(100).round(1)
TT = TT.reindex(['Saudável'] + ORDER + [NC]).dropna(how='all')
table(TT.reset_index(), 'Tipologia de sinal por estrato (% das sessões, vogal /a/ normal)',
      note='Regra operacional: Tipo III se ≥30% de quadros não vozeados ou ≥20% de quebras de voz; '
           'Tipo I se <5% não vozeados, nenhuma quebra e <1% de saltos de oitava no f0; demais = Tipo II. '
           'Medidas de perturbação são consideradas confiáveis em sinais Tipo I (e com cautela no Tipo II).')
TT.to_csv(os.path.join(TAB_DIR, 'tipologia_por_estrato.csv'))

t1_eff = {k: rbc(SM.loc[(SM.label == 0) & (SM.tipo_sinal == 'Tipo I'), c],SM.loc[(SM.label == 1) & (SM.tipo_sinal == 'Tipo I'), c])
          for k, c in PRAAT.items()}
CMP = pd.DataFrame({'todas as sessões': EFF['Todos'], 'só sinais Tipo I': pd.Series(t1_eff)}).round(3)
pct_t1 = (SM.loc[(SM.label == 1) & SM.tipo_sinal.notna(), 'tipo_sinal'] == 'Tipo I').mean()
table(CMP.reset_index().rename(columns={'index': 'parâmetro'}),
      'Efeitos recalculados só em sinais Tipo I',
      note=f'{pct_t1:.1%} das sessões patológicas são Tipo I. Se os efeitos de jitter se mantêm aqui, '
           'a liderança deles não é artefato de sinais aperiódicos.')
CMP.to_csv(os.path.join(TAB_DIR, 'efeito_tipo_I.csv'))


## Idade e sexo

In [ ]:
#@title 9 · Controle de idade e sexo real (substitui o proxy de 165 Hz)
# ---------- 9a. efeitos dentro de cada sexo ----------
rows = {}
for sx, lab in (('F', 'mulheres'), ('M', 'homens')):
    h = SM[(SM.label == 0) & (SM.sex == sx)]; p = SM[(SM.label == 1) & (SM.sex == sx)]
    rows[f'{lab} (n={len(h)}+{len(p)})'] = {k: rbc(h[c], p[c]) for k, c in PRAAT.items()}
SEX = pd.DataFrame(rows); SEX.insert(0, 'todos', EFF['Todos'])
a0 = SEX['todos']
robust = SEX.iloc[:, 1:].apply(lambda col: (np.sign(col) == np.sign(a0)) & (col.abs() >= a0.abs() / 2)).all(axis=1)
SEX['robusto'] = np.where(robust, 'sim', 'não')
table(SEX.round(3).reset_index().rename(columns={'index': 'parâmetro'}),
      'Efeitos estratificados pelo sexo registrado na SVD',
      note='Mesmo critério de robustez do artigo: mesmo sinal e pelo menos metade da magnitude nos dois estratos. '
           'Esta tabela substitui a estratificação por f0 a 165 Hz e dispensa a citação pedida pelo revisor.')
SEX.round(4).to_csv(os.path.join(TAB_DIR, 'efeito_por_sexo.csv'))

# ---------- 9b. pareamento 1:1 por idade dentro do sexo ----------
CALIPER = 3.0
rng = np.random.default_rng(SEED)
pairs = []
for sx in ('F', 'M'):
    hp = SM[(SM.label == 0) & (SM.sex == sx) & SM.age.notna()].copy()
    pp = SM[(SM.label == 1) & (SM.sex == sx) & SM.age.notna()].sample(frac=1, random_state=SEED)
    avail = hp.set_index('svd_session')['age'].to_dict()
    for _, r in pp.iterrows():
        if not avail: break
        k, a = min(avail.items(), key=lambda kv: abs(kv[1] - r.age))
        if abs(a - r.age) <= CALIPER:
            pairs.append((r.svd_session, k)); avail.pop(k)
matched_ids = {s for pr in pairs for s in pr}
MM = SM[SM.svd_session.isin(matched_ids)]
smd = (MM.loc[MM.label == 1, 'age'].mean() - MM.loc[MM.label == 0, 'age'].mean()) / MM.age.std()

def _boot_sess(sub):
    d = sub.reset_index(drop=True); g = {k: v.index.values for k, v in d.groupby('svd_speaker')}
    units = np.array(list(g)); r = np.random.default_rng(SEED); bs = []
    for _ in range(N_BOOT):
        dd = d.loc[np.concatenate([g[u] for u in r.choice(units, len(units))])]
        if dd.label.nunique() == 2: bs.append(roc_auc_score(dd.label, dd.prob_sess))
    return roc_auc_score(d.label, d.prob_sess), *np.percentile(bs, [2.5, 97.5])

auc_all = _boot_sess(SM); auc_mm = _boot_sess(MM)
se_mm = (MM.loc[MM.label == 1, 'prob_sess'] >= THR_SESS).mean()
sp_mm = (MM.loc[MM.label == 0, 'prob_sess'] < THR_SESS).mean()
eff_mm = {k: rbc(MM.loc[MM.label == 0, c], MM.loc[MM.label == 1, c]) for k, c in PRAAT.items()}
AGE = pd.DataFrame({'amostra completa': EFF['Todos'], 'pareada por idade': pd.Series(eff_mm)}).round(3)

panel('Pareamento por idade (1:1, dentro do sexo, caliper de 3 anos)',
      kpis=[('pares', f'{len(pairs):,}'),
            ('idade saudáveis', f'{MM.loc[MM.label == 0, "age"].mean():.1f} ± {MM.loc[MM.label == 0, "age"].std():.1f}'),
            ('idade patológicos', f'{MM.loc[MM.label == 1, "age"].mean():.1f} ± {MM.loc[MM.label == 1, "age"].std():.1f}'),
            ('diferença padronizada', f'{smd:+.3f}'),
            ('AUC completa', fmt_ci(*auc_all)), ('AUC pareada', fmt_ci(*auc_mm)),
            ('Se / Sp pareada', f'{se_mm:.3f} / {sp_mm:.3f}')],
      note='As probabilidades são as mesmas do CV agrupado por falante (o modelo nunca viu idade). '
           'Avaliar no subconjunto pareado responde se a discriminação sobrevive quando idade e sexo estão balanceados.')
table(AGE.reset_index().rename(columns={'index': 'parâmetro'}), 'Efeitos na amostra completa e na amostra pareada por idade')
AGE.to_csv(os.path.join(TAB_DIR, 'efeito_pareado_idade.csv'))
pd.DataFrame(pairs, columns=['sessao_patologica', 'sessao_saudavel']).to_csv(
    os.path.join(TAB_DIR, 'pares_idade.csv'), index=False)

# ---------- 9c. quanto cada parâmetro varia com a idade entre os saudáveis ----------
hh = SM[SM.label == 0]
AGEC = pd.DataFrame({k: stats.spearmanr(hh.age, hh[c], nan_policy='omit') for k, c in PRAAT.items()},
                    index=['rho', 'p']).T
AGEC['rho'] = AGEC['rho'].astype(float).round(3); AGEC['p'] = AGEC['p'].astype(float).map(lambda v: f'{v:.1e}')
table(AGEC.reset_index().rename(columns={'index': 'parâmetro'}),
      'Correlação idade × parâmetro entre os saudáveis (Spearman)',
      note='Mostra o efeito da idade sobre as próprias medidas acústicas, ponto levantado pelo revisor.')
AGEC.to_csv(os.path.join(TAB_DIR, 'idade_vs_parametros_saudaveis.csv'))


## O classificador aprendeu idade?

Na rodada v1 a AUC caiu de 0,842 para 0,694 na amostra pareada por idade, enquanto os efeitos univariados quase não mudaram. Este bloco testa se a queda vem da composição de gravidade, de pistas acústicas de idade usadas como atalho pelo classificador, ou de discriminação real menor em idades comparáveis.

In [ ]:
#@title 9b · A queda na amostra pareada: o classificador aprendeu idade?
# A AUC cai de forma acentuada no pareamento por idade, mas os efeitos univariados quase não mudam.
# Três verificações separam as explicações possíveis.

# ---------- (a) a amostra pareada tem outra composição de gravidade? ----------
comp = pd.DataFrame({
    'todos os patológicos': SM[SM.label == 1].estrato.value_counts(normalize=True),
    'patológicos pareados': MM[MM.label == 1].estrato.value_counts(normalize=True)}).mul(100).round(1)
comp = comp.reindex(ORDER + [NC]).dropna(how='all')
auc_s0_full = roc_auc_score(pd.concat([SM[SM.label == 0].label, SM[SM.estrato == ORDER[0]].label]),
                            pd.concat([SM[SM.label == 0].prob_sess, SM[SM.estrato == ORDER[0]].prob_sess]))
mm_s0 = MM[(MM.label == 0) | (MM.estrato == ORDER[0])]
auc_s0_mm = roc_auc_score(mm_s0.label, mm_s0.prob_sess) if mm_s0.label.nunique() == 2 else np.nan
table(comp.reset_index().rename(columns={'index': 'estrato'}),
      'Composição de gravidade: amostra completa × pareada (% das sessões patológicas)',
      note=f'AUC só no estrato S0 contra saudáveis: amostra completa {auc_s0_full:.3f}, pareada {auc_s0_mm:.3f}. '
           'Se a queda persiste dentro do mesmo estrato, ela não se explica por gravidade.')
comp.to_csv(os.path.join(TAB_DIR, 'gravidade_completa_vs_pareada.csv'))

# ---------- (b) as features acústicas codificam idade? (só saudáveis) ----------
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold
hs = df[df.label == 0].reset_index(drop=True)
pred_age = np.full(len(hs), np.nan)
for tr, te in GroupKFold(N_OUTER).split(hs, groups=hs.svd_speaker):
    m = Pipeline([('imp', SimpleImputer(strategy='median')),
                  ('reg', XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.05,
                                       tree_method='hist', device=DEVICE, random_state=SEED))])
    m.fit(hs.loc[tr, FEAT_COLS].values.astype(np.float32), hs.loc[tr, 'age'].values)
    pred_age[te] = m.predict(hs.loc[te, FEAT_COLS].values.astype(np.float32))
hs['pred_age'] = pred_age
hsess = hs.groupby('svd_session')[['age', 'pred_age']].mean()
rho_age = stats.spearmanr(hsess.age, hsess.pred_age).correlation
mae_age = (hsess.age - hsess.pred_age).abs().mean()
mae_base = (hsess.age - hsess.age.median()).abs().mean()

# ---------- (c) XGBoost treinado sem poder usar idade como atalho ----------
# Pesos que igualam a distribuição de idade × sexo dos saudáveis à dos patológicos dentro de cada
# fold de treino (faixas de 5 anos, só na região de suporte comum). O modelo deixa de ganhar
# discriminação com pistas acústicas de idade.
AGE_BIN = 5
df['age_bin'] = (df.age // AGE_BIN).astype('Int64').astype(str) + '_' + df.sex.astype(str)
sess_bins = df.groupby('svd_session').agg(b=('age_bin', 'first'), label=('label', 'first'))
cnt = pd.crosstab(sess_bins.b, sess_bins.label)
support = set(cnt[(cnt.get(0, 0) >= 3) & (cnt.get(1, 0) >= 3)].index)
in_sup = df.age_bin.isin(support).values

def age_weights(idx):
    d = df.iloc[idx]
    w = np.ones(len(d))
    sb = d.groupby(['svd_session']).agg(b=('age_bin', 'first'), label=('label', 'first'))
    ph = sb[sb.label == 1].b.value_counts(normalize=True)
    hh = sb[sb.label == 0].b.value_counts(normalize=True)
    ratio = (ph / hh).replace([np.inf], np.nan).fillna(0)
    hmask = d.label.values == 0
    w[hmask] = d.age_bin[hmask].map(ratio).fillna(0).values
    w[hmask] *= hmask.sum() / max(w[hmask].sum(), 1e-9)   # mantém o peso total da classe
    return w

cache_w = os.path.join(CACHE_DIR, f'oof_xgb_pesos_idade{CACHE_TAG}.parquet')
_cw = pd.read_parquet(cache_w) if os.path.exists(cache_w) and not FORCE_RERUN else None
if _cw is not None and len(_cw) == len(df):
    prob_w = _cw['prob'].values
else:
    prob_w = np.full(len(df), np.nan)
    idx_sup = np.where(in_sup)[0]
    g_sup = df.svd_speaker.astype(str).values[idx_sup]
    outer = StratifiedGroupKFold(N_OUTER, shuffle=True, random_state=SEED)
    for k, (tr_, te_) in enumerate(outer.split(idx_sup, y[idx_sup], g_sup)):
        tr, te = idx_sup[tr_], idx_sup[te_]
        w = age_weights(tr)
        inner = list(StratifiedGroupKFold(N_INNER, shuffle=True, random_state=SEED)
                     .split(X[tr], y[tr], df.svd_speaker.astype(str).values[tr]))
        pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                         ('clf', XGBClassifier(tree_method='hist', device=DEVICE, eval_metric='logloss', verbosity=0,
                                               scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
                                               random_state=SEED, n_jobs=-1 if DEVICE == 'cpu' else 1))])
        gs = GridSearchCV(pipe, XGB_GRID, scoring='roc_auc', cv=inner, n_jobs=1, refit=True)
        gs.fit(X[tr], y[tr], clf__sample_weight=w)
        prob_w[te] = gs.predict_proba(X[te])[:, 1]
        print(f'[pesos idade] fold {k + 1}/{N_OUTER}  AUC={roc_auc_score(y[te], prob_w[te]):.3f}')
    pd.DataFrame({'prob': prob_w}).to_parquet(cache_w)

df['prob_w'] = prob_w
pw = df.groupby('svd_session')['prob_w'].mean()
SM['prob_w'] = SM.svd_session.map(pw); MM = SM[SM.svd_session.isin(matched_ids)]
# Pares completos: as duas sessões do par têm previsão dos dois modelos (o modelo com pesos só prevê
# dentro do suporte comum). Toda comparação de modelos na amostra pareada usa estes mesmos pares.
_pw = SM.set_index('svd_session')['prob_w']
PAIRS_MODEL = [(p_, h_) for p_, h_ in pairs if pd.notna(_pw.get(p_)) and pd.notna(_pw.get(h_))]
MMs = SM[SM.svd_session.isin({s_ for pr in PAIRS_MODEL for s_ in pr})]
panel('Amostra pareada usada na comparação de modelos', kind='info',
      kpis=[('pares formados', len(pairs)), ('pares completos (os dois no suporte comum)', len(PAIRS_MODEL)),
            ('pares excluídos', len(pairs) - len(PAIRS_MODEL)),
            ('patológicos pareados sem AVQI', int(MMs[(MMs.label == 1)].avqi.isna().sum()))],
      note='O modelo com pesos de idade × sexo só gera previsão para sessões em faixas de 5 anos × sexo com pelo menos '
           '3 sessões de cada classe. Para que os dois modelos sejam comparados nas mesmas sessões e o pareamento se '
           'mantenha, a avaliação de modelos usa apenas os pares em que as duas sessões têm previsão. Os efeitos '
           'univariados continuam usando todos os pares.')
sup_s = SM[SM.prob_w.notna()]
def _auc(d, col):
    d = d[d[col].notna()]
    return roc_auc_score(d.label, d[col]) if d.label.nunique() == 2 else np.nan

AGEM = pd.DataFrame([
    {'modelo': 'original (sem controle de idade)', 'suporte comum': _auc(sup_s, 'prob_sess'),
     'amostra pareada': _auc(MMs, 'prob_sess')},
    {'modelo': 'pesos de idade × sexo no treino', 'suporte comum': _auc(sup_s, 'prob_w'),
     'amostra pareada': _auc(MMs, 'prob_w')}]).round(3)

panel('As features acústicas carregam informação de idade',
      kpis=[('Spearman idade prevista × real (saudáveis)', f'{rho_age:.2f}'),
            ('erro médio (anos)', f'{mae_age:.1f}'), ('erro só com a mediana', f'{mae_base:.1f}'),
            ('sessões no suporte comum', f'{sup_s.svd_session.nunique():,}')],
      note='Regressão XGBoost da idade a partir das 79 features, só em saudáveis, CV agrupado por falante. '
           'Se a idade é previsível pela voz, um classificador treinado com grupos de idades tão diferentes '
           'pode usar essas pistas como atalho.')
table(AGEM, 'AUC (nível de sessão) com e sem controle de idade no treino',
      note='Suporte comum: faixas de 5 anos × sexo com pelo menos 3 sessões em cada classe. '
           'Se o modelo com pesos recupera AUC na amostra pareada, a queda vinha do atalho de idade; '
           'se não recupera, a discriminação real em idades comparáveis é a da amostra pareada.')
AGEM.to_csv(os.path.join(TAB_DIR, 'auc_controle_idade.csv'), index=False)


## Tabelas finais para o artigo

Desempenho por estrato de gravidade com idade controlada de duas formas (pesos no treino e pareamento na avaliação), Tabela 3 revisada e Tabela 1 recontada. São estas as tabelas que entram no manuscrito.

In [ ]:
#@title 9c · Tabelas finais para o artigo: desempenho por estrato com idade controlada e Tabela 3 revisada
def _youden(d, col):
    d = d[d[col].notna()]
    f, t, th = roc_curve(d.label, d[col]); j = np.argmax(t - f)
    return th[j]

MODELS = {'original': ('prob_sess', _youden(SM, 'prob_sess')),
          'pesos de idade × sexo': ('prob_w', _youden(SM, 'prob_w'))}
SAMPLES = {'completa (suporte comum)': SM[SM.prob_w.notna()], 'pareada por idade e sexo': MMs}   # pares completos

def _strat_metrics(base, patho, col, thr):
    hh = base[(base.label == 0) & base[col].notna()]
    pp = patho[patho[col].notna()]
    d = pd.concat([hh, pp]).reset_index(drop=True)
    if d.label.nunique() < 2 or len(pp) < 5:
        return None
    g = {k: v.index.values for k, v in d.groupby('svd_speaker')}
    units = np.array(list(g)); r = np.random.default_rng(SEED); ba, bs = [], []
    for _ in range(N_BOOT):
        dd = d.loc[np.concatenate([g[u] for u in r.choice(units, len(units))])]
        if dd.label.nunique() == 2:
            ba.append(roc_auc_score(dd.label, dd[col]))
            bs.append((dd.loc[dd.label == 1, col] >= thr).mean())
    auc = roc_auc_score(d.label, d[col]); se = (pp[col] >= thr).mean(); sp = (hh[col] < thr).mean()
    return {'saudáveis': len(hh), 'patológicos': len(pp),
            'AUC': fmt_ci(auc, *np.percentile(ba, [2.5, 97.5])),
            'Sensibilidade': fmt_ci(se, *np.percentile(bs, [2.5, 97.5])),
            'Especificidade': round(sp, 3), '_auc': auc, '_se': se}

rows = []
for mname, (col, thr) in MODELS.items():
    for sname, base in SAMPLES.items():
        for st in ['Todos'] + ORDER:
            patho = base[base.label == 1] if st == 'Todos' else base[base.estrato == st]
            m = _strat_metrics(base, patho, col, thr)
            if m:
                rows.append({'modelo': mname, 'amostra': sname, 'estrato': st, **m})
FIN = pd.DataFrame(rows, columns=['modelo', 'amostra', 'estrato', 'saudáveis', 'patológicos', 'AUC', 'Sensibilidade', 'Especificidade', '_auc', '_se']) if not rows else pd.DataFrame(rows)
for mname in MODELS:
    table(FIN[FIN.modelo == mname].drop(columns=['modelo', '_auc', '_se']),
          f'Desempenho por estrato de gravidade · modelo {mname}',
          note=f'Limiar de Youden do modelo na amostra completa = {MODELS[mname][1]:.3f}, mantido fixo na amostra pareada. '
               'Cada estrato patológico é comparado com todos os saudáveis da mesma amostra. '
               f'IC 95% por bootstrap de falantes reais ({N_BOOT} reamostragens).')
FIN.drop(columns=['_auc', '_se']).to_csv(os.path.join(TAB_DIR, 'TABELA_desempenho_estrato_idade.csv'), index=False)

_fw = FIN[FIN.modelo == 'pesos de idade × sexo']
piv = (_fw.pivot(index='estrato', columns='amostra', values='_se') if len(_fw) else pd.DataFrame())
piv = piv.reindex(index=['Todos'] + ORDER, columns=list(SAMPLES))
def _draw_fin(fig, ax, dark):
    x = np.arange(len(piv)); w = .38
    cols = list(piv.columns)
    ax.bar(x - w / 2, piv[cols[0]], w, color='#6E6E6E' if dark else '#9A9A9A', label=cols[0])
    ax.bar(x + w / 2, piv[cols[1]], w, color=OR, label=cols[1])
    for i, (a, b) in enumerate(zip(piv[cols[0]], piv[cols[1]])):
        for xx, v in ((i - w / 2, a), (i + w / 2, b)):
            if not np.isnan(v):
                ax.text(xx, v + .015, f'{v:.2f}', ha='center', fontsize=8, color=TXT if dark else '#222')
    ax.set_xticks(x); ax.set_xticklabels(piv.index, rotation=12)
    ax.set_ylim(0, 1.1); ax.set_ylabel('Sensibilidade')
    ax.set_title('Sensibilidade por estrato · modelo com controle de idade')
    leg = ax.legend(fontsize=8, frameon=False, loc='upper left')
    for t in leg.get_texts(): t.set_color(TXT if dark else '#222')
if piv.notna().any().any():
    figure(_draw_fin, 'fig_sensibilidade_estrato_pareada', (8.5, 4.4))

# ---------- Tabela 3 revisada ----------
def _holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for k, i in enumerate(o):
        run = max(run, min(1, (m - k) * p[i])); adj[i] = run
    return adj

def _mw_p(h, p):
    h, p = pd.Series(h).dropna(), pd.Series(p).dropna()
    return stats.mannwhitneyu(h, p).pvalue if len(h) > 2 and len(p) > 2 else np.nan

def _med_iqr(v):
    v = pd.Series(v).dropna()
    return f'{v.median():.3g} [{v.quantile(.25):.3g}; {v.quantile(.75):.3g}]'

T1 = SM[SM.tipo_sinal == 'Tipo I']
T3 = pd.DataFrame({
    'saudáveis mediana [IQR]': pd.Series({k: _med_iqr(SM.loc[SM.label == 0, c]) for k, c in PRAAT.items()}),
    'patológicos mediana [IQR]': pd.Series({k: _med_iqr(SM.loc[SM.label == 1, c]) for k, c in PRAAT.items()}),
    'r completa': EFF['Todos'],
    'p Holm completa': pd.Series(_holm([_mw_p(SM.loc[SM.label == 0, c], SM.loc[SM.label == 1, c]) for c in PRAAT.values()]), index=list(PRAAT)),
    'r pareada': pd.Series(eff_mm),
    'p Holm pareada': pd.Series(_holm([_mw_p(MM.loc[MM.label == 0, c], MM.loc[MM.label == 1, c]) for c in PRAAT.values()]), index=list(PRAAT)),
    'r só Tipo I': pd.Series(t1_eff),
    'robusto ao sexo': SEX['robusto']})
for c in ('r completa', 'r pareada', 'r só Tipo I'):
    T3[c] = T3[c].astype(float).round(3)
for c in ('p Holm completa', 'p Holm pareada'):
    T3[c] = T3[c].map(lambda v: f'{v:.1e}')
T3 = T3.reindex(EFF['Todos'].abs().sort_values(ascending=False).index)
table(T3.reset_index().rename(columns={'index': 'parâmetro'}),
      'Tabela 3 revisada (unidade = sessão, sem duplicatas)',
      note='r = correlação rank-bisserial (negativo = maior no grupo patológico). Pareada: 1:1 por idade dentro do sexo, '
           'caliper de 3 anos. Tipo I: só sinais quase periódicos. Holm sobre os parâmetros da tabela em cada coluna.')
T3.to_csv(os.path.join(TAB_DIR, 'TABELA3_revisada.csv'))

T1n = (SM.groupby('diagnosis')
         .agg(sessões=('svd_session', 'size'), falantes=('svd_speaker', 'nunique'),
              gravações=('n_rec', 'sum'), mulheres_pct=('sex', lambda s: 100 * (s == 'F').mean()),
              idade_média=('age', 'mean'), idade_DP=('age', 'std'))
         .round(1).reset_index())
table(T1n, 'Tabela 1 recontada (sessões únicas, falantes reais, idade e sexo)')
T1n.to_csv(os.path.join(TAB_DIR, 'TABELA1_recontada.csv'), index=False)


## Validação perceptiva

O índice acústico precisa ser ancorado em julgamento clínico. Este bloco sorteia uma amostra estratificada e gera as fichas e os áudios, com códigos cegos, para duas fonoaudiólogas.

In [ ]:
#@title 10 · Pacote cego para avaliação perceptiva (GRBAS / CAPE-V) por duas fonoaudiólogas
import shutil, zipfile
PKG = os.path.join(RATE_DIR, 'pacote_avaliadoras'); AUD_DIR = os.path.join(PKG, 'audio')
KEY = os.path.join(RATE_DIR, 'CHAVE_NAO_ENVIAR.csv')

if os.path.exists(KEY) and not FORCE_RERUN:
    key = pd.read_csv(KEY)
    panel('Pacote de avaliação já existe', f'<p>Usando a chave salva com {len(key)} itens. '
          'Para refazer o sorteio, rode com FORCE_RERUN = True.</p>')
else:
    elig = sess[sess.avqi.notna()].copy()
    # uma sessão por falante, para a avaliação não repetir vozes
    elig = elig.sample(frac=1, random_state=SEED).drop_duplicates('svd_speaker')
    pat = elig[elig.label == 1]
    # alocação proporcional por estrato × diagnóstico, com no mínimo 3 por célula quando possível
    cells = pat.groupby(['estrato', 'diagnosis'])
    alloc = (cells.size() / len(pat) * N_RATE_PATH).round().astype(int).clip(lower=3)
    alloc = np.minimum(alloc, cells.size())
    pick_p = pd.concat([g.sample(int(alloc[k]), random_state=SEED) for k, g in cells if alloc[k] > 0])
    pick_h = elig[elig.label == 0].sample(min(N_RATE_HEALTHY, (elig.label == 0).sum()), random_state=SEED)
    items = pd.concat([pick_p, pick_h])
    dups = items.sample(max(1, int(round(RATE_DUP_FRAC * len(items)))), random_state=SEED + 1)
    items = pd.concat([items.assign(repeticao=0), dups.assign(repeticao=1)]).sample(frac=1, random_state=SEED + 2)
    codes = np.random.default_rng(SEED).choice(np.arange(1000, 9999), len(items), replace=False)
    items['codigo'] = [f'V{c}' for c in codes]
    key = items[['codigo', 'svd_session', 'svd_speaker', 'label', 'diagnosis', 'estrato', 'avqi', 'repeticao']]
    key.to_csv(KEY, index=False)

    os.makedirs(AUD_DIR, exist_ok=True)
    for _, r in key.iterrows():
        s = int(r.svd_session)
        shutil.copy(_to_wav(AUD[(s, 'a_n')], f'{s}-a_n'), os.path.join(AUD_DIR, f'{r.codigo}_vogal_a.wav'))
        shutil.copy(_to_wav(AUD[(s, 'phrase')], f'{s}-phrase'), os.path.join(AUD_DIR, f'{r.codigo}_frase.wav'))

    sheet = pd.DataFrame({'codigo': sorted(key.codigo),
                          'G (0-3)': '', 'R (0-3)': '', 'B (0-3)': '', 'A (0-3)': '', 'S (0-3)': '',
                          'CAPE-V gravidade geral (0-100)': '', 'observacoes': ''})
    instr = pd.DataFrame({'Instruções': [
        'Ouça a vogal /a/ e a frase de cada código, nesta ordem, com fones de ouvido.',
        'Atribua G, R, B, A e S de 0 (normal) a 3 (grave). Se possível, marque também a gravidade geral do CAPE-V (0 a 100).',
        'Não consulte a outra avaliadora. Alguns itens se repetem de propósito.',
        'Salve o arquivo com seu nome e devolva como avaliador1.xlsx ou avaliador2.xlsx.']})
    for i in (1, 2):
        with pd.ExcelWriter(os.path.join(PKG, f'ficha_avaliador{i}.xlsx')) as w:
            instr.to_excel(w, sheet_name='Instrucoes', index=False)
            sheet.to_excel(w, sheet_name='Avaliacao', index=False)
    zpath = os.path.join(RATE_DIR, 'pacote_avaliacao_perceptiva.zip')
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(PKG):
            for f in files:
                z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), PKG))
    comp = key.groupby(['estrato']).size().rename('itens').reset_index()
    table(comp, 'Pacote de avaliação perceptiva gerado',
          note=f'{len(key)} itens ({int(key.repeticao.sum())} repetições para confiabilidade intra-avaliador). '
               f'Zip para enviar às avaliadoras: <code>{zpath}</code>. A chave fica só com você: <code>{KEY}</code>.')


In [ ]:
#@title 11 · Análise das avaliações perceptivas (roda quando as fichas voltarem)
from sklearn.metrics import cohen_kappa_score
RET = os.path.join(RATE_DIR, 'retorno')
os.makedirs(RET, exist_ok=True)
files = {i: os.path.join(RET, f'avaliador{i}.xlsx') for i in (1, 2)}

if not all(os.path.exists(f) for f in files.values()):
    panel('Aguardando as fichas das avaliadoras',
          f'<p>Coloque as fichas preenchidas em <code>{RET}</code> com os nomes '
          '<code>avaliador1.xlsx</code> e <code>avaliador2.xlsx</code> e rode este bloco de novo.</p>'
          '<p>Ele calcula: concordância entre e intra-avaliadoras (kappa ponderado), distribuição do G na '
          'amostra patológica, correlação AVQI × G e sensibilidade do XGBoost por grau G0–G3.</p>')
else:
    key = pd.read_csv(KEY)
    R = []
    for i, f in files.items():
        x = pd.read_excel(f, sheet_name='Avaliacao')
        gcol = next(c for c in x.columns if str(c).startswith('G'))
        R.append(x[['codigo', gcol]].rename(columns={gcol: f'G{i}'}))
    RR = key.merge(R[0], on='codigo').merge(R[1], on='codigo').dropna(subset=['G1', 'G2'])
    RR[['G1', 'G2']] = RR[['G1', 'G2']].astype(int)
    base = RR[RR.repeticao == 0]
    k_inter = cohen_kappa_score(base.G1, base.G2, weights='quadratic')
    intra = []
    for i in (1, 2):
        rep = RR[RR.repeticao == 1].set_index('svd_session')[f'G{i}']
        orig = base.set_index('svd_session')[f'G{i}']
        common = rep.index.intersection(orig.index)
        if len(common) > 2:
            intra.append(cohen_kappa_score(orig[common], rep[common], weights='quadratic'))
    base = base.assign(G=np.rint(base[['G1', 'G2']].mean(axis=1)).astype(int))
    base = base.merge(sess[['svd_session', 'prob_sess']], on='svd_session', how='left')
    rho_g = stats.spearmanr(base.avqi, base.G)
    pat = base[base.label == 1]
    byG = (pat.assign(det=(pat.prob_sess >= THR_SESS).astype(int))
              .groupby('G').agg(sessões=('det', 'size'), sensibilidade=('det', 'mean'),
                                AVQI_mediana=('avqi', 'median')).reset_index())
    byG['% da amostra patológica'] = 100 * byG['sessões'] / byG['sessões'].sum()
    panel('Confiabilidade e validade concorrente',
          kpis=[('kappa ponderado entre', f'{k_inter:.2f}'),
                ('kappa intra (média)', f'{np.mean(intra):.2f}' if intra else 'n/d'),
                ('Spearman AVQI × G', f'{rho_g.correlation:.2f}'),
                ('itens avaliados', len(base))])
    table(byG.round(3), 'Gravidade perceptiva (G consensual) e sensibilidade do XGBoost',
          note='A proporção de G0–G1 na amostra patológica responde diretamente à pergunta do revisor sobre '
               'o peso de casos leves; a sensibilidade por grau mostra o que o instrumento perde nesses casos.')
    byG.to_csv(os.path.join(TAB_DIR, 'sensibilidade_por_G.csv'), index=False)


## Resumo

In [ ]:
#@title 12 · Resumo para a carta de resposta ao revisor
import json
pf = PERF.set_index('estrato')
s0 = 100 * (P.estrato == ORDER[0]).mean()
s12 = 100 * P.estrato.isin(ORDER[:3]).mean()
res = {
    'sessoes': int(len(sess)), 'falantes_reais': int(sess.svd_speaker.nunique()),
    'auc_grupo_sessao': round(a_s[0], 3), 'auc_grupo_falante': round(a_p[0], 3),
    'idade_saudavel': f'{sess.loc[sess.label == 0, "age"].mean():.1f} ± {sess.loc[sess.label == 0, "age"].std():.1f}',
    'idade_patologico': f'{sess.loc[sess.label == 1, "age"].mean():.1f} ± {sess.loc[sess.label == 1, "age"].std():.1f}',
    'auc_idade_sozinha': round(auc_age, 3),
    'auc_pareada_idade': round(auc_mm[0], 3), 'pares_idade': len(pairs),
    'limiar_avqi_p95_saudaveis': round(float(AVQI_THR), 2),
    'pct_patologicos_faixa_normal_avqi': round(s0, 1),
    'pct_patologicos_ate_moderada': round(s12, 1),
    'sens_por_estrato': {k: round(v, 3) for k, v in pf['_se'].items()},
    'auc_por_estrato': {k: round(v, 3) for k, v in pf['_auc'].items()},
    'pct_tipo_I_patologicos': round(100 * pct_t1, 1),
    'sessoes_em_varias_pastas': int(len(multi_diag)), 'gravacoes_duplicadas': n_dup_rec,
    'rho_idade_prevista_saudaveis': round(float(rho_age), 2),
    'auc_pareada_modelo_original': float(AGEM.loc[0, 'amostra pareada']),
    'auc_pareada_modelo_pesos_idade': float(AGEM.loc[1, 'amostra pareada']),
    'gravacoes_apos_remover_duplicatas': int(len(df)),
    'final_pesos_idade': {f'{r.amostra} | {r.estrato}': {'AUC': r.AUC, 'Se': r.Sensibilidade, 'Sp': r.Especificidade}
                          for r in FIN[FIN.modelo == 'pesos de idade × sexo'].itertuples()}}
json.dump(res, open(os.path.join(OUT_DIR, 'resumo_revisao.json'), 'w'), ensure_ascii=False, indent=2)

li = ''.join(f'<li>{t}</li>' for t in [
    f'A amostra tem {res["sessoes"]:,} sessões de {res["falantes_reais"]:,} falantes reais; '
    f'com CV agrupado pelo falante real a AUC é {res["auc_grupo_falante"]} (agrupando por sessão: {res["auc_grupo_sessao"]}).',
    f'Idade: saudáveis {res["idade_saudavel"]} anos, patológicos {res["idade_patologico"]}; '
    f'a idade sozinha atinge AUC {res["auc_idade_sozinha"]}. Na amostra pareada por idade e sexo '
    f'({res["pares_idade"]} pares) a AUC acústica é {res["auc_pareada_idade"]}.',
    f'{res["pct_patologicos_faixa_normal_avqi"]}% das sessões patológicas têm AVQI dentro da faixa dos saudáveis '
    f'(≤ {res["limiar_avqi_p95_saudaveis"]}): a amostra é dominada por quadros leves ou compensados.',
    'Sensibilidade por estrato: ' + ', '.join(f'{k} = {v}' for k, v in res['sens_por_estrato'].items()) + '.',
    f'{res["pct_tipo_I_patologicos"]}% das sessões patológicas são sinais Tipo I, nos quais jitter e shimmer são confiáveis.',
    f'{res["sessoes_em_varias_pastas"]} sessões aparecem em mais de uma pasta diagnóstica ({res["gravacoes_duplicadas"]} gravações duplicadas na matriz).',
    f'As features prevêem a idade dos saudáveis (ρ = {res["rho_idade_prevista_saudaveis"]}). Na amostra pareada, '
    f'AUC {res["auc_pareada_modelo_original"]} com o modelo original e {res["auc_pareada_modelo_pesos_idade"]} com pesos de idade no treino.'] +
    [f'Modelo com controle de idade, {k}: AUC {v["AUC"]}, sensibilidade {v["Se"]}'
     for k, v in res['final_pesos_idade'].items() if k.startswith('pareada')])
panel('Números-chave para a resposta', f'<ul>{li}</ul>', kind='good',
      note=f'Tabelas em {TAB_DIR} · figuras (300 dpi, fundo branco) em {FIG_DIR} · resumo em resumo_revisao.json')


## Figuras e tabelas no formato do artigo

Gera as Figuras 1, 2, 7, 8 e 9 (PNG 300 dpi e PDF) e as Tabelas 1, 2, 3, 4, 8 e 9 em LaTeX, prontas para o main.tex. Tudo vai para `svd_data/revisao_artigo`, a mesma pasta usada pelo notebook das Tabelas 5 a 7.

In [ ]:
#@title Estilo das figuras do artigo e geração de LaTeX
import html as _html
import matplotlib as mpl
from matplotlib.patches import Rectangle

ART_DIR = os.path.join(SVD_DIR, 'revisao_artigo')        # pasta única para os dois notebooks
ART_FIG = os.path.join(ART_DIR, 'figuras')                 # PNG 300 dpi + PDF vetorial
ART_TEX = os.path.join(ART_DIR, 'latex')                   # tabelas e ambientes de figura prontos para o main.tex
for _d in (ART_DIR, ART_FIG, ART_TEX):
    os.makedirs(_d, exist_ok=True)
FIG_SUBDIR = 'figuras_v4'                                  # pasta usada no \includegraphics do main.tex

PAPER = {'navy': '#08306B', 'blue': '#2171B5', 'mid': '#6BAED6', 'light': '#C6DBEF',
         'orange': '#D95F02', 'lorange': '#FDAE6B', 'green': '#1B9E77', 'gray': '#4D4D4D',
         'lgray': '#BDBDBD', 'pink': '#C51B7D', 'gold': '#E6AB02'}
PAPER_RC = {'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.titlesize': 8.5, 'axes.labelsize': 8,
            'xtick.labelsize': 7.5, 'ytick.labelsize': 7.5, 'legend.fontsize': 7, 'legend.frameon': False,
            'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': '#555555',
            'axes.grid': True, 'grid.color': '#E6E6E6', 'grid.linestyle': '--', 'grid.linewidth': .6,
            'axes.axisbelow': True, 'figure.facecolor': 'white', 'axes.facecolor': 'white',
            'savefig.facecolor': 'white', 'figure.dpi': 110}
DXN = {'healthy': 'Healthy', 'hyperfunctional_dysphonia': 'Hyperfunctional dysphonia',
       'vocal_fold_paralysis': 'Vocal fold paralysis', 'laryngitis': 'Laryngitis',
       'polyp': 'Vocal fold polyp', 'nodule': 'Vocal fold nodule'}
DX_ORDER = ['hyperfunctional_dysphonia', 'vocal_fold_paralysis', 'laryngitis', 'polyp', 'nodule']

def dxname(d):
    return DXN.get(str(d), str(d).replace('_', ' ').capitalize())

def plab(k):
    """Rótulo de parâmetro com f0 em subscrito (matplotlib e LaTeX)."""
    return str(k).replace('Mean f0', 'Mean $f_0$').replace('f0 SD', '$f_0$ SD').replace('f0 parameters', '$f_0$ parameters')

def tlab(k):
    return plab(k).replace('%', '\\%')

def stars(p):
    return '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else 'ns'

def tex_num(v, d=3, sign=False):
    if v is None or not np.isfinite(v):
        return '--'
    return f'${v:+.{d}f}$' if sign else f'${v:.{d}f}$'

def tex_p(p):
    if p is None or not np.isfinite(p):
        return '--'
    if p >= 0.001:
        return f'{p:.3f}'
    m, e = f'{p:.1e}'.split('e')
    return f'${m}\\times10^{{{int(e)}}}$'

def tex_pm(s):
    return str(s).replace(' ± ', ' $\\pm$ ').replace('±', '$\\pm$')

FIG_TEX, TAB_TEX = {}, {}

def paper_figure(name, draw, label, caption, nrows=1, ncols=1, figsize=(7.2, 3.0),
                 width_ratios=None, width='\\textwidth'):
    """Desenha no estilo do artigo, salva PNG (300 dpi) e PDF, e escreve o ambiente LaTeX com a legenda."""
    with mpl.rc_context(PAPER_RC):
        fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False,
                                 gridspec_kw={'width_ratios': width_ratios} if width_ratios else None)
        draw(fig, list(axes.ravel()))
        fig.tight_layout()
        fig.savefig(os.path.join(ART_FIG, name + '.png'), dpi=300, bbox_inches='tight')
        fig.savefig(os.path.join(ART_FIG, name + '.pdf'), bbox_inches='tight')
        plt.show()
    tex = ('\\begin{figure*}[htbp]\n\\centering\n'
           f'\\includegraphics[width={width}]{{{FIG_SUBDIR}/{name}.png}}\n'
           f'\\caption{{{caption}}}\n\\label{{{label}}}\n\\end{{figure*}}\n')
    FIG_TEX[name] = tex
    open(os.path.join(ART_TEX, f'{name}.tex'), 'w').write(tex)

def latex_table(name, caption, label, header, rows, colfmt, note, font='\\small', tabcolsep=None, header_tex=None):
    """Tabela booktabs no formato do main.tex. Em rows, a string 'MID' insere um \\midrule."""
    L = ['\\begin{table*}[htbp]', '\\centering', f'\\caption{{{caption}}}', f'\\label{{{label}}}', font]
    if tabcolsep:
        L.append(f'\\setlength{{\\tabcolsep}}{{{tabcolsep}}}')
    L += [f'\\begin{{tabular}}{{{colfmt}}}', '\\toprule', header_tex or (' & '.join(header) + ' \\\\'), '\\midrule']
    for r in rows:
        L.append('\\midrule' if r == 'MID' else ' & '.join(map(str, r)) + ' \\\\')
    L += ['\\bottomrule', '\\end{tabular}', '', '\\vspace{2mm}', '\\begin{minipage}{0.95\\textwidth}',
          '\\footnotesize', '\\raggedright', note, '\\end{minipage}', '\\end{table*}', '']
    tex = '\n'.join(L)
    TAB_TEX[name] = tex
    open(os.path.join(ART_TEX, f'{name}.tex'), 'w').write(tex)
    prev = pd.DataFrame([r for r in rows if r != 'MID'], columns=header)
    table(prev, f'{name} · {caption}', note=f'LaTeX salvo em latex/{name}.tex')

def show_latex(title, parts, fname):
    txt = '\n'.join(parts)
    open(os.path.join(ART_TEX, fname), 'w').write(txt)
    panel(title, f'<pre style="white-space:pre-wrap;font-size:11px;color:#EDEDED;background:#1C1C1C;'
                 f'padding:10px;border-radius:6px;max-height:420px;overflow:auto">{_html.escape(txt)}</pre>',
          note=f'Arquivo completo: {os.path.join(ART_TEX, fname)}. Cada tabela e figura também tem seu próprio .tex na mesma pasta.')

panel('Figuras e tabelas no formato do artigo',
      f'<p>PNG em 300 dpi e PDF em <code>{ART_FIG}</code>; LaTeX em <code>{ART_TEX}</code>.</p>')


In [ ]:
#@title 13 · Figuras 1, 2, 7, 8 e 9 e Tabelas 1, 2, 3, 4, 8 e 9 no formato do artigo
from sklearn.metrics import precision_recall_curve

N_H, N_P = int((SM.label == 0).sum()), int((SM.label == 1).sum())
N_SESS, N_REC = len(SM), len(df)
N_PAIRS = len(pairs)
NS_TEX, NR_TEX = f'{N_SESS:,}'.replace(',', '{,}'), f'{N_REC:,}'.replace(',', '{,}')   # milhar no modo matemático
KORD = EFF['Todos'].abs().sort_values(ascending=False).index.tolist()      # ordem por magnitude
JL = PRAAT.get('Jitter local (%)')
JL_SCALE = 100.0 if JL and SM.loc[SM.label == 0, JL].median() < 0.05 else 1.0   # a matriz guarda a fração
def _sc(k):
    return JL_SCALE if k == 'Jitter local (%)' else 1.0

PFULL = pd.Series(_holm([_mw_p(SM.loc[SM.label == 0, c], SM.loc[SM.label == 1, c]) for c in PRAAT.values()]), index=list(PRAAT))
PMM = pd.Series(_holm([_mw_p(MM.loc[MM.label == 0, c], MM.loc[MM.label == 1, c]) for c in PRAAT.values()]), index=list(PRAAT))
c_w = [c for c in SEX.columns if str(c).startswith('mulheres')][0]
c_m = [c for c in SEX.columns if str(c).startswith('homens')][0]
n_w = re.findall(r'\d+', c_w); n_m = re.findall(r'\d+', c_m)
ST_LAB = {ORDER[0]: 'S0', ORDER[1]: 'S1', ORDER[2]: 'S2', ORDER[3]: 'S3'}
ST_LONG = {ORDER[0]: 'S0 (within healthy range)', ORDER[1]: 'S1 (mild)', ORDER[2]: 'S2 (moderate)', ORDER[3]: 'S3 (marked)'}
N_ST = P.estrato.value_counts()

# ======================= Figura 1 =======================
def _fig1(fig, ax):
    ax = ax[0]; ks = KORD[::-1]; v = EFF['Todos'].abs()[ks].values
    ax.barh(range(len(ks)), v, height=.7, edgecolor='white',
            color=[PAPER['navy'] if x >= .3 else PAPER['mid'] for x in v])
    for i, (k, x) in enumerate(zip(ks, v)):
        ax.text(x + .006, i, f'{x:.3f} {stars(PFULL[k])}', va='center', fontsize=7)
    for t, l in ((.1, 'small'), (.3, 'medium')):
        ax.axvline(t, ls=':', color=PAPER['gray'], lw=.8)
        ax.text(t, len(ks) - .35, l, ha='center', va='bottom', fontsize=6.5, color=PAPER['gray'])
    ax.set_yticks(range(len(ks))); ax.set_yticklabels([plab(k) for k in ks])
    ax.set_xlim(0, max(.5, v.max() + .09)); ax.set_ylim(-.6, len(ks) + .1); ax.grid(axis='y', visible=False)
    ax.set_xlabel('|r| rank-biserial correlation (healthy vs. pathological)')
    ax.set_title(f'Binary discrimination by Praat acoustic parameter, session level (n = {N_SESS:,})')
paper_figure('Fig1_rankbiserial_session', _fig1, 'fig:effect_sizes',
             f'Absolute rank-biserial correlation for each clinical parameter at the session level '
             f'($n={NS_TEX}$: {N_H} healthy, {N_P} pathological), each session represented by the median of its productions, '
             'ordered by magnitude. Dotted lines mark the conventional thresholds for small (0.1) and medium (0.3) effects. '
             'Asterisks denote significance after Holm--Bonferroni correction (*** $p<0.001$).',
             figsize=(6.4, 3.4), width='0.85\\linewidth')

# ======================= Figura 2 =======================
_norm = df['pitch'].astype(str).str.lower().isin(['n', 'normal']) if 'pitch' in df else pd.Series(True, index=df.index)
RREC = {}
for k, c in PRAAT.items():
    m = _norm if k == 'f0 SD (Hz)' else pd.Series(True, index=df.index)
    RREC[k] = rbc(df.loc[m & (df.label == 0), c], df.loc[m & (df.label == 1), c])
RREC = pd.Series(RREC)
SIGN = np.sign(EFF['Todos'])
def _aligned(s):
    return (pd.Series(s)[KORD] * SIGN[KORD]).values   # positivo = mesma direção do efeito geral

def _fig2(fig, axes):
    ks = KORD[::-1]; yy = np.arange(len(ks)); h = .38
    spec = [((RREC, f'Recordings (n = {N_REC:,})', PAPER['light']), (EFF['Todos'], f'Session medians (n = {N_SESS:,})', PAPER['navy']),
             '(A) Unit of analysis'),
            ((SEX[c_w], f'Women ({n_w[0]} + {n_w[1]})', PAPER['orange']), (SEX[c_m], f'Men ({n_m[0]} + {n_m[1]})', PAPER['blue']),
             '(B) Recorded sex'),
            ((EFF['Todos'], f'Full sample (n = {N_SESS:,})', PAPER['navy']), (AGE['pareada por idade'], f'Age- and sex-matched ({N_PAIRS} pairs)', PAPER['orange']),
             '(C) Age matching')]
    for ax, (a, b, title) in zip(axes, spec):
        va = _aligned(a[0])[::-1]; vb = _aligned(b[0])[::-1]
        ax.barh(yy + h / 2, va, h, color=a[2], edgecolor='white', label=a[1])
        ax.barh(yy - h / 2, vb, h, color=b[2], edgecolor='white', label=b[1])
        ax.axvline(0, color='#555', lw=.7); ax.axvline(.3, ls=':', color=PAPER['gray'], lw=.7)
        ax.set_yticks(yy); ax.set_yticklabels([plab(k) for k in ks] if ax is axes[0] else [])
        ax.set_xlim(min(-.15, np.nanmin([va.min(), vb.min()]) - .03), max(.65, np.nanmax([va.max(), vb.max()]) + .05)); ax.grid(axis='y', visible=False)
        ax.set_title(title); ax.set_xlabel('Aligned |r|')
        ax.legend(loc='upper center', bbox_to_anchor=(.5, -.16), fontsize=6.3, handlelength=1.2)
paper_figure('Fig2_effect_unit_sex_age', _fig2, 'fig:rankbiserial',
             f'Rank-biserial effect size of each clinical parameter, signed so that positive values follow the direction '
             f'of the overall effect (aligned $|r|$); a bar to the left of zero marks a reversal. (A) Computed over recordings ($n={NR_TEX}$) and over session medians ($n={NS_TEX}$). '
             '(B) Within women and men, using the sex recorded in the SVD metadata (healthy + pathological sessions). '
             f'(C) In the full sample and in the sample matched one to one on sex and age within 3 years ({N_PAIRS} pairs). '
             'The dotted line marks a medium effect (0.3).',
             ncols=3, figsize=(7.2, 4.0), width_ratios=[1.25, 1, 1])

# ======================= Figura 7 =======================
col_w, thr_w = MODELS['pesos de idade × sexo']
_samples = [('Full sample, common age–sex range', SM[SM[col_w].notna()], PAPER['navy']),
            ('Age- and sex-matched sample', MMs, PAPER['orange'])]
OP7 = {}
def _fig7(fig, axes):
    a, b = axes
    for name, d, c in _samples:
        f, t, _ = roc_curve(d.label, d[col_w]); auc = roc_auc_score(d.label, d[col_w])
        se = (d.loc[d.label == 1, col_w] >= thr_w).mean(); sp = (d.loc[d.label == 0, col_w] < thr_w).mean()
        OP7[name] = (auc, se, sp, d.label.mean(), len(d))
        a.plot(f, t, color=c, lw=1.6, label=f'{name} (AUC {auc:.3f})')
        a.scatter([1 - sp], [se], s=34, color=c, edgecolor='black', zorder=5)
        a.annotate(f'Se {se:.2f}, Sp {sp:.2f}', (1 - sp, se), xytext=(8, -12), textcoords='offset points', fontsize=6.5, color=c)
        pr, rc, _ = precision_recall_curve(d.label, d[col_w])
        b.plot(rc, pr, color=c, lw=1.6, label=name)
        b.axhline(d.label.mean(), ls=':', color=c, lw=1)
    a.plot([0, 1], [0, 1], ls=':', color='#888', lw=.8)
    a.set_xlabel('1 − specificity'); a.set_ylabel('Sensitivity'); a.set_title('(A) ROC curve, age-reweighted XGBoost (sessions)')
    a.legend(loc='lower right', fontsize=6.5)
    b.set_xlabel('Sensitivity (recall)'); b.set_ylabel('Positive predictive value'); b.set_title('(B) Precision–recall curve')
    b.set_ylim(.3, 1.02); b.legend(loc='lower left', fontsize=6.5)
paper_figure('Fig7_operating_point_sessions', _fig7, 'fig:clinical_operating_point',
             'Session-level operating characteristics of the age-reweighted XGBoost model, in the full sample within the '
             f'common age and sex range and in the {len(PAIRS_MODEL)} age- and sex-matched pairs with predictions from both models. (A) Receiver operating characteristic curves; '
             "the marked points use the threshold set by Youden's $J$ in the full sample, kept fixed in the matched sample. "
             '(B) Precision--recall curves; dotted lines mark the prevalence of each sample.',
             ncols=2, figsize=(7.2, 3.1), width='0.90\\textwidth')

# ======================= Figura 8 =======================
_dx = [d for d in DX_ORDER if d in set(P.diagnosis)] + sorted(set(P.diagnosis) - set(DX_ORDER))
def _fig8(fig, ax):
    ax = ax[0]
    data = [H.avqi.values] + [P.loc[P.diagnosis == d, 'avqi'].values for d in _dx]
    labs = [f'Healthy (n = {len(H)})'] + [f'{dxname(d)} (n = {len(v)})' for d, v in zip(_dx, data[1:])]
    xmax = np.nanpercentile(np.concatenate(data), 99.7) + .4
    for (lo_, hi_, s, al) in ((AVQI_THR, t1, 'S1', .25), (t1, t2, 'S2', .45), (t2, xmax, 'S3', .7)):
        ax.axvspan(lo_, hi_, color=PAPER['light'], alpha=al, lw=0)
        ax.text((lo_ + min(hi_, xmax)) / 2, len(data) + .55, s, ha='center', fontsize=7, color=PAPER['navy'])
    ax.text(AVQI_THR / 2, len(data) + .55, 'S0', ha='center', fontsize=7, color=PAPER['navy'])
    ypos = np.arange(len(data), 0, -1)
    bp = ax.boxplot(data, positions=ypos, vert=False, patch_artist=True, showfliers=False, widths=.55)
    for i, bx in enumerate(bp['boxes']):
        bx.set_facecolor(PAPER['lgray'] if i == 0 else PAPER['mid']); bx.set_edgecolor('#333'); bx.set_alpha(.9)
    for k in ('medians', 'whiskers', 'caps'):
        for l in bp[k]: l.set_color('#222')
    r_ = np.random.default_rng(0)
    for yv, v in zip(ypos, data):
        ax.scatter(v, yv + r_.uniform(-.2, .2, len(v)), s=2.5, alpha=.35, color=PAPER['navy'], lw=0)
    ax.axvline(AVQI_THR, ls='--', color=PAPER['orange'], lw=1.2, label=f'P{HEALTHY_PCTL} of healthy controls = {AVQI_THR:.2f}')
    ax.axvline(2.43, ls=':', color='#222', lw=1, label='Published cutoff = 2.43')
    ax.set_yticks(ypos); ax.set_yticklabels(labs); ax.set_xlim(0, xmax); ax.set_ylim(.4, len(data) + 1)
    ax.grid(axis='y', visible=False); ax.set_xlabel('AVQI v03.01 (higher = poorer voice quality)')
    ax.legend(loc='lower right', fontsize=6.5)
paper_figure('Fig8_avqi_distribution', _fig8, 'fig:avqi_distribution',
             'Distribution of AVQI v03.01 by diagnostic group, one point per session. The dashed line marks the 95th '
             f'percentile of healthy controls ({AVQI_THR:.2f}), the upper limit of stratum S0; the dotted line marks the published '
             f'cutoff of 2.43~\\cite{{ref28}}. Shaded bands show strata S1 to S3, the tertiles of pathological sessions above the healthy range '
             f'(limits {t1:.2f} and {t2:.2f}).',
             figsize=(6.6, 3.4), width='0.85\\linewidth')

# ======================= Figura 9 =======================
pf = PERF.set_index('estrato')
fw = FIN[(FIN.modelo == 'pesos de idade × sexo') & (FIN.amostra == 'pareada por idade e sexo')].set_index('estrato')
fo = FIN[(FIN.modelo == 'original') & (FIN.amostra == 'pareada por idade e sexo')].set_index('estrato')
SP_FULL = sp_h; SP_MM = float(fw.loc['Todos', 'Especificidade']) if 'Todos' in fw.index else np.nan
def _fig9(fig, axes):
    a, b, c = axes
    M = EFF.loc[KORD, ORDER].abs().values
    im = a.imshow(M, cmap='Blues', vmin=0, vmax=1, aspect='auto')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            a.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=6.3, color='white' if M[i, j] > .55 else '#222')
    a.set_xticks(range(4)); a.set_xticklabels([f'{ST_LAB[s]}\nn = {N_ST.get(s, 0)}' for s in ORDER], fontsize=6.8)
    a.set_yticks(range(len(KORD))); a.set_yticklabels([plab(k) for k in KORD]); a.grid(False)
    a.set_title('(A) |r| against healthy sessions')
    x = np.arange(4); w = .38
    s_full = [pf.loc[s, '_se'] if s in pf.index else np.nan for s in ORDER]
    s_mm = [fw.loc[s, '_se'] if s in fw.index else np.nan for s in ORDER]
    b.bar(x - w / 2, s_full, w, color=PAPER['navy'], label=f'Full sample, original model (Sp {SP_FULL:.2f})')
    b.bar(x + w / 2, s_mm, w, color=PAPER['orange'], label=f'Matched, age-reweighted model (Sp {SP_MM:.2f})')
    for xx, v in list(zip(x - w / 2, s_full)) + list(zip(x + w / 2, s_mm)):
        if np.isfinite(v): b.text(xx, v + .015, f'{v:.2f}', ha='center', fontsize=5.4, rotation=90, va='bottom')
    b.set_xticks(x); b.set_xticklabels([ST_LAB[s] for s in ORDER]); b.set_ylim(0, 1.2); b.set_ylabel('Sensitivity')
    b.set_title('(B) Sensitivity by AVQI stratum'); b.legend(loc='upper center', bbox_to_anchor=(.5, -.12), fontsize=6, handlelength=1)
    qs = cc['sensibilidade'].values
    c.bar(range(len(qs)), qs, .6, color=[PAPER['light'], PAPER['mid'], PAPER['blue'], PAPER['navy']][:len(qs)])
    for i, v in enumerate(qs): c.text(i, v + .02, f'{v:.2f}', ha='center', fontsize=6.3)
    c.set_xticks(range(len(qs))); c.set_xticklabels(['Q1\nhighest', 'Q2', 'Q3', 'Q4\nlowest'][:len(qs)])
    c.set_ylim(0, 1.18); c.set_xlabel('Quartile of sentence CPPS'); c.set_title('(C) Independent severity index')
    for ax_ in (b, c): ax_.grid(axis='x', visible=False)
paper_figure('Fig9_severity_effects_sensitivity', _fig9, 'fig:severity',
             '(A) Absolute rank-biserial correlation between healthy sessions and each AVQI severity stratum of pathological '
             'sessions. (B) Session-level sensitivity by stratum, for the original model in the full sample and for the '
             'age-reweighted model in the age- and sex-matched sample, at the operating points of Table~\\ref{tab:severity_performance}. '
             '(C) Sensitivity of the original model by quartile of CPPS measured on the connected speech sentence alone, '
             'which the classifier never sees.',
             ncols=3, figsize=(7.2, 3.6), width_ratios=[1.3, 1.1, .75])

# ======================= Tabela 1 =======================
def _t1row(name, g):
    return [name, f'{len(g):,}', f'{g.svd_speaker.nunique():,}', f'{int(g.n_rec.sum()):,}',
            f'{100 * (g.sex == "F").mean():.1f}', f'{g.age.mean():.1f} $\\pm$ {g.age.std():.1f}']
rows = [_t1row('Healthy', SM[SM.label == 0])] + [_t1row(dxname(d), SM[(SM.label == 1) & (SM.diagnosis == d)]) for d in _dx]
rows += ['MID', _t1row('All pathological', SM[SM.label == 1]), _t1row('Total', SM)]
latex_table('Tabela1', 'Corpus composition after eligibility screening and removal of duplicated sessions.', 'tab:corpus',
            ['Diagnostic group', 'Sessions', 'Speakers', 'Recordings', 'Women (\\%)', 'Age, years (M $\\pm$ SD)'], rows, 'lrrrrr',
            f'\\textit{{Note.}} A session is one visit of one speaker. In total, {len(multi)} speakers recorded more than one session '
            f'and {len(both)} contributed sessions to both classes, so speakers per group do not sum to the total. The {len(multi_diag)} '
            'sessions listed under two diagnoses in the SVD were counted once, under the first applicable diagnosis in the order paralysis, '
            'polyp, nodule, laryngitis, hyperfunctional dysphonia. Age at the date of recording, from the SVD metadata. '
            'Pitch conditions: normal, high, low, and a rise and fall in pitch.')

# ======================= Tabela 2 (só a legenda muda) =======================
T2 = r'''\begin{table*}[!t]
\centering
\caption{Composition of the librosa feature set. Series: number of frame-level time series of each type. MFCC, mel-frequency cepstral coefficient; $\Delta$, first temporal derivative of each MFCC (velocity); $\Delta\Delta$, second temporal derivative (acceleration). Statistics retained: summary computed over the frames of a recording to obtain one value per series. Features: series $\times$ statistics.}
\label{tab:librosa_composition}
\small
\begin{tabular}{lccc}
\toprule
Component & Series & Statistics retained & Features \\
\midrule
Cepstral coefficients (base) & 13 & mean, standard deviation & 26 \\
Cepstral coefficients ($\Delta$) & 13 & mean & 13 \\
Cepstral coefficients ($\Delta\Delta$) & 13 & mean & 13 \\
\addlinespace
\multicolumn{3}{l}{\itshape Cepstral subtotal} & \itshape 52 \\
\addlinespace
Spectral centroid, bandwidth, roll-off, flatness & 4 & mean, standard deviation & 8 \\
Root-mean-square energy, zero-crossing rate & 2 & mean, standard deviation & 4 \\
\midrule
Total & 45 & --- & 64 \\
\bottomrule
\end{tabular}
\end{table*}
'''
TAB_TEX['Tabela2'] = T2; open(os.path.join(ART_TEX, 'Tabela2.tex'), 'w').write(T2)

# ======================= Tabela 3 =======================
def _mq(v, k):
    v = pd.Series(v).dropna() * _sc(k)
    return f'{v.median():.3g} [{v.quantile(.25):.3g}; {v.quantile(.75):.3g}]'
rows = []
for k in KORD:
    c = PRAAT[k]
    rows.append([tlab(k) + ('$^{\\dagger}$' if k == 'f0 SD (Hz)' else ''),
                 _mq(SM.loc[SM.label == 0, c], k), _mq(SM.loc[SM.label == 1, c], k),
                 tex_num(EFF.loc[k, 'Todos'], sign=True), tex_p(PFULL[k]),
                 tex_num(AGE.loc[k, 'pareada por idade'], sign=True), tex_p(PMM[k]),
                 tex_num(t1_eff[k], sign=True), tex_num(SEX.loc[k, c_w], sign=True), tex_num(SEX.loc[k, c_m], sign=True)])
latex_table('Tabela3', 'Clinical acoustic parameters: full sample, age- and sex-matched sample, Type I signals and sex strata '
            '(Mann--Whitney $U$, rank-biserial $r$).', 'tab:clinical_parameters',
            ['Parameter', 'Healthy Md [Q1; Q3]', 'Pathological Md [Q1; Q3]', '$r$ (full)', '$p$ (Holm)',
             '$r$ (matched)', '$p$ (Holm)', '$r$ (Type I)', '$r$ women', '$r$ men'], rows, 'lccccccccc',
            header_tex=('& & & \\multicolumn{2}{c}{Full sample} & \\multicolumn{2}{c}{Matched} & Type I & Women & Men \\\\\n'
                        '\\cmidrule(lr){4-5}\\cmidrule(lr){6-7}\n'
                        'Parameter & Healthy & Pathological & $r$ & $p$ (Holm) & $r$ & $p$ (Holm) & $r$ & $r$ & $r$ \\\\'),
            note=f'\\textit{{Note.}} Unit of analysis is the session, represented by the median of its productions '
            f'({N_H} healthy, {N_P} pathological). Matched: {N_PAIRS} pairs matched one to one on sex and age within 3 years. '
            'Healthy and pathological columns give the median [first quartile; third quartile]. Type I: sessions whose normal-pitch /a/ is a nearly periodic signal. Negative $r$ indicates higher values in the '
            'pathological group. Holm--Bonferroni correction within each column. $^{\\dagger}$Normal-pitch recordings only.',
            font='\\scriptsize', tabcolsep='3pt')

# ======================= Tabela 4 =======================
NORM = [('Jitter local (%)', 'lt', 1.04, '$<1.04$', 'Baken \\& Orlikoff~\\cite{ref3}', 3),
        ('Shimmer local', 'lt', 0.14, '$<0.14$', 'Baken \\& Orlikoff~\\cite{ref3}', 4),
        ('Shimmer (dB)', 'lt', 0.35, '$<0.35$', 'Baken \\& Orlikoff~\\cite{ref3}', 3),
        ('HNR (dB)', 'gt', 20.0, '$>20.0$', 'Yumoto et al.~\\cite{ref4}', 2),
        ('CPPS (dB)', 'gt', 14.0, '$>14.0$', 'Heman-Ackah et al.~\\cite{ref18}', 2)]
rows, flagged = [], []
for k, kind, lim, ltxt, ref, d in NORM:
    if k not in PRAAT:
        continue
    cells = []
    for lab_ in (0, 1):
        v = SM.loc[SM.label == lab_, PRAAT[k]].dropna() * _sc(k)
        beyond = (v > lim) if kind == 'lt' else (v < lim)
        crosses = (v.mean() > lim) if kind == 'lt' else (v.mean() < lim)
        cells.append((f'${v.mean():.{d}f} \\pm {v.std():.{d}f}$' + ('$^{*}$' if crosses else ''), 100 * beyond.mean()))
        if crosses: flagged.append(k)
    rows.append([tlab(k), cells[0][0], cells[1][0], ltxt, f'{cells[0][1]:.1f} / {cells[1][1]:.1f}', ref])
latex_table('Tabela4', 'Acoustic parameters against published normative ranges.', 'tab:normative_ranges',
            ['Parameter', 'Healthy (M $\\pm$ SD)', 'Pathological (M $\\pm$ SD)', 'Limit',
             'Beyond, H / P (\\%)', 'Reference'], rows, 'lccccl',
            f'\\textit{{Note.}} Session medians ({N_H} healthy, {N_P} pathological sessions). '
            '$^{*}$Mean beyond the normative limit. The last numeric column gives the percentage of healthy and of '
            'pathological sessions whose value lies beyond the limit.')

# ======================= Tabela 8 =======================
D8 = DIST.set_index('grupo')
_names = {'Saudável': 'Healthy', 'Todos patológicos': 'All pathological'}
rows = []
for g in ['Saudável'] + _dx + ['Todos patológicos']:
    if g not in D8.index:
        continue
    r = D8.loc[g]
    st = ['--'] * 4 if g == 'Saudável' else [f'{r[s]:.1f}' for s in ORDER]
    if g == 'Todos patológicos':
        rows.append('MID')
    rows.append([_names.get(g, dxname(g)), f'{int(r["n"])}', f'{r["AVQI mediana"]:.1f} ({r["IQR"]:.1f})',
                 f'{r["% > 2.43"]:.1f}', *st])
latex_table('Tabela8', 'Acoustic severity by diagnostic group (AVQI v03.01).', 'tab:severity_composition',
            ['Group', 'Sessions', 'AVQI Md (IQR)', '$>2.43$ (\\%)', 'S0 (\\%)', 'S1 (\\%)', 'S2 (\\%)', 'S3 (\\%)'],
            rows, 'lrcccccc',
            f'\\textit{{Note.}} Higher AVQI indicates poorer voice quality. S0: AVQI $\\le {AVQI_THR:.2f}$, the 95th percentile '
            f'of healthy controls. S1 to S3: tertiles of the remaining pathological sessions (limits {t1:.2f} and {t2:.2f}). '
            '2.43: published cutoff for AVQI v03.01~\\cite{ref28}, shown for reference. '
            f'AVQI could not be computed for {int(sess.avqi.isna().sum())} sessions without a sentence recording.')

# ======================= Tabela 9 =======================
def _ci(sv):
    m = re.match(r'([\d.]+) \[([\d.]+); ([\d.]+)\]', str(sv))
    return f'{m.group(1)} ({m.group(2)}--{m.group(3)})' if m else str(sv)
rows = []
for s in ['Todos'] + ORDER:
    lab_ = 'All pathological' if s == 'Todos' else ST_LONG[s]
    rows.append([lab_, f'{pf.loc[s, "_auc"]:.3f}' if s in pf.index else '--', f'{pf.loc[s, "_se"]:.3f}' if s in pf.index else '--',
                 f'{fo.loc[s, "_auc"]:.3f}' if s in fo.index else '--',
                 _ci(fw.loc[s, 'AUC']) if s in fw.index else '--',
                 f'{fw.loc[s, "_se"]:.3f}' if s in fw.index else '--',
                 f'{int(fw.loc[s, "patológicos"])}' if s in fw.index else '--'])
    if s == 'Todos':
        rows.append('MID')
n_h_mm = int(fw.loc['Todos', 'saudáveis']) if 'Todos' in fw.index else 0
latex_table('Tabela9', 'Session-level discrimination by severity stratum, with and without control of age.',
            'tab:severity_performance',
            ['Stratum', 'Full, original: AUC', 'Full: sensitivity', 'Matched, original: AUC',
             'Matched, age-reweighted: AUC (95\\% CI)', 'Matched, age-reweighted: sensitivity', 'Matched pathological sessions'],
            rows, 'lcccccc',
            header_tex=('& \\multicolumn{2}{c}{Full sample, original model} & \\multicolumn{4}{c}{Age- and sex-matched sample} \\\\\n'
                        '\\cmidrule(lr){2-3}\\cmidrule(lr){4-7}\n'
                        'Stratum & AUC & Sensitivity & AUC, original & AUC, reweighted (95\\% CI) & Sensitivity, reweighted & $n$ \\\\'),
            note=f'\\textit{{Note.}} Matched sample: the {len(PAIRS_MODEL)} of {N_PAIRS} age- and sex-matched pairs in which both sessions '
            f'lie within the common age-by-sex support and therefore have predictions from both models; the same pairs are used '
            f'for the original and the age-reweighted model. $n$: pathological sessions of each stratum in these pairs '
            f'(strata require an AVQI value). Each stratum is compared with all healthy sessions of the same sample (full sample: {N_H}; '
            f'matched: {n_h_mm}). Probabilities averaged over the recordings of each session, '
            'from five-fold nested cross-validation grouped by speaker. Operating point fixed at Youden\'s $J$ of the full '
            f'sample: specificity {SP_FULL:.3f} (original model, full sample) and {SP_MM:.3f} (age-reweighted model, matched sample). '
            f'Confidence intervals from 2{{,}}000 bootstrap resamples of speakers.',
            font='\\footnotesize')

order_tex = ['Tabela1', 'Tabela2', 'Tabela3', 'Tabela4', 'Fig1_rankbiserial_session', 'Fig2_effect_unit_sex_age',
             'Fig7_operating_point_sessions', 'Tabela8', 'Tabela9', 'Fig8_avqi_distribution', 'Fig9_severity_effects_sensitivity']
show_latex('LaTeX pronto: corpus, efeitos, idade, gravidade e ponto de operação',
           [TAB_TEX.get(n) or FIG_TEX.get(n) for n in order_tex], 'gravidade_idade_tabelas_figuras.tex')
panel('Ponto de operação da Figura 7', kind='good',
      kpis=[(n, f'AUC {v[0]:.3f} · Se {v[1]:.2f} · Sp {v[2]:.2f}') for n, v in OP7.items()],
      note=f'Limiar de Youden do modelo com pesos de idade na amostra completa = {thr_w:.3f}, mantido na amostra pareada.')
